v62 
- convert codebase for RL agent  
- headless scorer
- cross-sectional normalization
- discovery composite action
- temporal vectorizer

v61 
- **Verified all metric calculation with Excel** 
- Updated core.analyzer
- Replaced the `Result` pattern with exceptions and flattened the logic
- Refactored the `AlphaEngine` into a streamlined Orchestrator pattern
-  **Strict Date Logic:** No more "Time Travel" bugs.
-  **Dataclass Contracts:** No more "Magic String" typos or blind dictionaries.
-  **Exception Flow:** The `run` method is now a clean, readable story instead of a maze of `if/else` checks.
-  **Performance Workers:** Math is separated from orchestration.
- Ret_1d, explicitly turns division-by-zero results (`inf`) into `NaN`, and replace [np.inf, -np.inf] with np.nan



v60  
- Converted code from notebook to modular system.
- Fixed divide by zero warning from calculate_gain
- Added subtitle to subplots
- Added Volatility Regime plot


v59  
- Removed "nest" of if-statements in **AlphaEngine.run**
- Use **Result Pattern** to handle errors
- Change verify_analyzer_short and verify_analyzer_long gain calculation from simple return to logarithmic return
- Change calculate_gain from simple return to logarithmic return
- Remove bfill from calculate_gain to prevent backfill with future data
- Verify macro_df calculation


In [16]:
# 1. Enable Autoreload
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

def add_project_root_to_path():
    """Find notebooks_RLVR and add to sys.path."""
    current = Path.cwd()

    # Search upward for notebooks_RLVR folder
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            print(f"✓ Added to path: {path}")
            return path
        # Also check if notebooks_RLVR exists as child (for running from stocks/)
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            print(f"✓ Added to path: {candidate}")
            return candidate

    raise RuntimeError("Could not find notebooks_RLVR directory")


# Run once at notebook start
add_project_root_to_path()


# 2. Force reload cached modules (run this to refresh code changes)
modules_to_reload = [
    "core.analyzer",
    "core.auditor",
    "core.builder",
    "core.contracts",    
    "core.engine",
    "core.environment",
    "core.features",
    "core.logic",
    "core.paths",
    "core.performance",
    "core.quant",
    "core.result",
    "core.settings",
    "core.utils",
    "strategy.registry",
]

for mod in modules_to_reload:
    if mod in sys.modules:
        del sys.modules[mod]


# 3. Standard imports
import pandas as pd
import numpy as np
import multiprocessing
import gc

from IPython.display import display


# 4. Fresh imports (these will re-import from disk due to cache clearing above)
from core.quant import QuantUtils
# from core.analyzer import create_walk_forward_analyzer
from core.analyzer import create_walk_forward_analyzer, run_headless_simulation
from core.auditor import SystemAuditor as SA
from core.builder import ParallelFeatureBuilder, FeatureCubeStitcher
from core.contracts import FilterPack, EngineInput
from core.engine import AlphaEngine, AlphaCache
from core.environment import DiscoveryEnv
from core.features import generate_features
from core.logic import AlphaLogic, SelectionLogic
from core.paths import OUTPUT_DIR
from core.settings import GLOBAL_SETTINGS
from core.utils import SystemUtils as SU
from strategy.registry import METRIC_REGISTRY


# 5. Pandas display settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.precision", 4)


# 6. Load data path from .env
DATA_PATH_OHLCV = SU.load_env_from_root("DATA_PATH_OHLCV")
DATA_PATH_INDICES = SU.load_env_from_root("DATA_PATH_INDICES")
print(f"DATA_PATH_OHLCV: {DATA_PATH_OHLCV}")
print(f"DATA_PATH_INDICES: {DATA_PATH_INDICES}")

# # 7. Instantiate engine (customize DataFrames as needed)
# new_master_engine = AlphaEngine(
#     df_ohlcv=df_ohlcv,
#     features_df=features_df,
#     macro_df=macro_df,
#     df_close_wide=df_close_wide,
#     df_atrp_wide=df_atrp_wide,
#     df_trp_wide=df_trp_wide,
# )

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ Added to path: c:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR
NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output

DATA_PATH_OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
DATA_PATH_INDICES: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


In [11]:
#### Change path to frozen data ####
DATA_PATH_OHLCV = r"c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs_20260324.parquet"
DATA_PATH_INDICES = (
    r"c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices_20260324.parquet"
)
print(f"=== Overwrite DATA_PATHS ===")
print(f"DATA_PATH_INDICES: {DATA_PATH_INDICES}")
print(f"DATA_PATH_OHLCV: {DATA_PATH_OHLCV}")

=== Overwrite DATA_PATHS ===
DATA_PATH_INDICES: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices_20260324.parquet
DATA_PATH_OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs_20260324.parquet


In [ ]:
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
print(f"df_indices:\n{df_indices}")

df_indices:|n                   Adj Open  Adj High  Adj Low  Adj Close  Volume
Ticker Date                                                      
^AXJO  1992-11-22   1455.00   1455.00  1455.00    1455.00       0
       1992-11-23   1458.40   1458.40  1458.40    1458.40       0
       1992-11-24   1467.90   1467.90  1467.90    1467.90       0
       1992-11-25   1459.00   1459.00  1459.00    1459.00       0
       1992-11-26   1458.90   1458.90  1458.90    1458.90       0
...                     ...       ...      ...        ...     ...
^VIX3M 2026-03-18     25.00     26.57    24.82      26.56       0
       2026-03-19     27.96     28.03    25.19      25.54       0
       2026-03-20     26.07     28.61    25.91      27.43       0
       2026-03-23     25.60     26.59    24.73      26.10       0
       2026-03-24     26.86     27.20    25.79      26.56       0

[144848 rows x 5 columns]


In [ ]:
print(f"Takes about 1.5 minutes")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")
print(f"df_ohlcv:\n{df_ohlcv}")

Takes about 1.5 minutes
df_ohlcv:|n                   Adj Open  Adj High  Adj Low  Adj Close    Volume
Ticker Date                                                        
A      1999-11-18   27.1966   29.8864  23.9091    26.3000  74849975
       1999-11-19   25.6649   25.7023  23.7970    24.1333  18230876
       1999-11-22   24.6936   26.3000  23.9465    26.3000   7871813
       1999-11-23   25.4034   26.0759  23.9091    23.9091   7151083
       1999-11-24   23.9838   25.0672  23.9091    24.5442   5795948
...                     ...       ...      ...        ...       ...
ZWS    2026-03-18   45.0500   45.1200  44.2300    44.3400   1162000
       2026-03-19   43.6700   44.6200  43.4000    44.0400   1077900
       2026-03-20   44.2400   44.3300  43.0600    43.7900   3255800
       2026-03-23   45.0000   45.9600  44.4700    45.1600   1209900
       2026-03-24   45.0100   45.8100  44.6100    45.4700    825400

[9528174 rows x 5 columns]


In [50]:
df_ohlcv.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 9528174 entries, ('A', Timestamp('1999-11-18 00:00:00')) to ('ZWS', Timestamp('2026-03-24 00:00:00'))
Data columns (total 5 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Adj Open   float64
 1   Adj High   float64
 2   Adj Low    float64
 3   Adj Close  float64
 4   Volume     int64  
dtypes: float64(4), int64(1)
memory usage: 400.5+ MB


In [5]:
print(f"Takes about 3 minutes to generate_features")

features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

Takes about 3 minutes to generate_features
⚡ Generating Decoupled Features (Benchmark: SPY)...


In [ ]:
# _indices = df_indices.index.get_level_values(0).unique().tolist()
# display(_indices)
# df_indices.info()

# # print(f"df_ohlcv.head():\n {df_ohlcv.head()}\n")
# df_ohlcv.info()

In [ ]:
# print(f"features_df.info():\n{features_df.info()}\n")
# print(f"features_df.index.names:\n{features_df.index.names}\n")
# print(f"macro_df.info():\n{macro_df.info()}\n")
# print(f"macro_df.index.names:\n{macro_df.index.names}\n")

In [ ]:
# # Pre-flight Automated Audit Suite
# checks = [
#     SA.verify_math_integrity(),
#     SA.verify_ranking_integrity(),
#     SA.verify_vol_alignment_integrity(),
#     SA.verify_feature_engineering_integrity(),
# ]

# for check in checks:
#     icon = "✅" if check.ok else "🔥"
#     print(f"{icon} {check.msg}")
#     if not check.ok:
#         print("🛑 Critical Failure. System Halted.")
#         break

# print("=" * 85)
# # Separate verify_marco_engine output from intertwine with other outputs

# checks = [
#     SA.verify_macro_engine(
#         df_ohlcv=df_ohlcv,
#         df_indices=df_indices,
#         original_macro_df=macro_df,
#         settings=GLOBAL_SETTINGS,
#     ),
# ]

# for check in checks:
#     icon = "✅" if check.ok else "🔥"
#     print(f"{icon} {check.msg}")
#     if not check.ok:
#         print("🛑 Critical Failure. System Halted.")
#         break

In [14]:
print(
    "🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)"
)

# 1. Price Matrix
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)

# 2. Volatility Matrices (Unstack and Align)
# Using features_df (the first item from the tuple)
print("   - Unstacking ATRP...")
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)

print("   - Unstacking TRP...")
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

# 3. Handle Data Gaps (Sanitize the Wide Matrices)
if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

# Forward fill up to the limit, then fill remaining with the "Disaster Detection" value
df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pre-computation Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)
print("   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.")

🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)
   - Unstacking ATRP...
   - Unstacking TRP...
✅ Pre-computation Complete. Tickers: 1585, Days: 16164
   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.


In [ ]:
print(f"df_close_wide.info():\n{df_close_wide.info()}")
print(f"df_atrp_wide.info():\n{df_atrp_wide.info()}")
print(f"df_trp_wide.info():\n{df_trp_wide.info()}")

In [17]:
# 6. Instantiate engine (customize DataFrames as needed)
new_master_engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)

#### Run ParallelFeatureBuilder

In [19]:
# # --- THE MARATHON CONFIG ---
CHECKPOINT_DIR = "_alpha_cache_v1_checkpoints"
FINAL_FILE = "AlphaCache_Master_2025_2026.parquet"
# # I recommend leaving 2 cores for your PC.
# # If you have an 8-core CPU, it will use 6 for the Bake.
WORKER_COUNT = max(1, multiprocessing.cpu_count() - 2)

ParallelFeatureBuilder.run_marathon(
    master_engine=new_master_engine,
    # lookbacks=[21, 63, 189],
    lookbacks=[10, 21],
    start_date="2025-01-01",
    checkpoint_dir=CHECKPOINT_DIR,
    batch_size=50,
    num_workers=WORKER_COUNT,  # <--- TAME THE BEAST
)

# 2. THE STITCH (Run this when the progress bar hits 100%)
final_cache_df = FeatureCubeStitcher.assemble(CHECKPOINT_DIR, FINAL_FILE)

🚀 Parallel Bake: 306 days | Using 2 of 4 cores.
🖥️  2 cores reserved for your PC usage.


Baking AlphaCache:   0%|          | 0/306 [00:00<?, ?it/s]

✨ Marathon complete. All batches saved to _alpha_cache_v1_checkpoints
🧵 Stitching 7 batches...


Merging:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Final Master Cube Saved! Shape: (285767, 26)


In [ ]:
#############################
#############################

In [ ]:
import pandas as pd
import numpy as np
import os


class PostBuildVerifier:
    """
    Post-build verification suite for Z-score validation.
    Run this AFTER FeatureCubeStitcher.assemble() completes.
    """

    def __init__(self, final_cache_df):
        self.df = final_cache_df
        self.numeric_cols = [
            c for c in final_cache_df.columns if final_cache_df[c].dtype == "float64"
        ]
        self.verification_results = {}
        self.available_dates = final_cache_df.index.get_level_values("Date").unique()

    def generate_summary_report(self):
        """Generate comprehensive summary statistics."""
        print("\n" + "=" * 60)
        print("FINAL CACHE VERIFICATION REPORT")
        print("=" * 60)

        print(f"\n📐 Dataset Shape: {self.df.shape}")
        date_range = self.df.index.get_level_values("Date")
        print(f"📅 Date Range: {date_range.min()} to {date_range.max()}")
        print(
            f"🏷️  Unique Tickers: {self.df.index.get_level_values('Ticker').nunique()}"
        )
        print(f"📊 Numeric Features: {len(self.numeric_cols)}")

        # Overall statistics across all Z-scores
        all_values = self.df[self.numeric_cols].values.flatten()
        all_values = all_values[~np.isnan(all_values)]

        print(f"\n🌍 Global Statistics (all Z-scores pooled):")
        print(f"   Mean: {np.mean(all_values):.6f} (should be ~0)")
        print(f"   Std:  {np.std(all_values, ddof=1):.6f} (should be ~1)")
        print(f"   Min:  {np.min(all_values):.4f}")
        print(f"   Max:  {np.max(all_values):.4f}")
        print(
            f"   NaN%: {100 * np.isnan(self.df[self.numeric_cols].values).sum() / self.df[self.numeric_cols].size:.2f}%"
        )

        return pd.DataFrame(
            {
                "Metric": ["Mean", "Std", "Min", "Max"],
                "Value": [
                    np.mean(all_values),
                    np.std(all_values, ddof=1),
                    np.min(all_values),
                    np.max(all_values),
                ],
            }
        )

    def verify_cross_sectional_normality(self, sample_dates=None, tolerance=1e-10):
        """
        Verify that Z-scores are properly normalized per date.
        Mean should be ~0, Std should be ~1.
        """
        if sample_dates is None:
            # Auto-select 5 dates
            indices = np.linspace(0, len(self.available_dates) - 1, 5, dtype=int)
            sample_dates = [self.available_dates[i] for i in indices]

        print(f"\n🔍 Verifying {len(sample_dates)} sample dates...")
        results = []

        for date in sample_dates:
            date_slice = self.df.loc[date]
            row = {"Date": date}

            for col in self.numeric_cols:
                values = date_slice[col].dropna()
                if len(values) > 0:
                    row[f"{col}_mean"] = values.mean()
                    row[f"{col}_std"] = values.std(ddof=1)
                    row[f"{col}_count"] = len(values)

            results.append(row)

        verify_df = pd.DataFrame(results)

        # Check for violations
        mean_cols = [c for c in verify_df.columns if "_mean" in c]
        std_cols = [c for c in verify_df.columns if "_std" in c]

        max_mean_dev = verify_df[mean_cols].abs().max().max()
        max_std_dev = (verify_df[std_cols] - 1.0).abs().max().max()

        print(f"   Max |mean| deviation: {max_mean_dev:.2e} (tolerance: {tolerance})")
        print(f"   Max |std-1| deviation: {max_std_dev:.2e}")
        print(
            f"   Mean ≈ 0 check: {'✅ PASS' if max_mean_dev < tolerance else '❌ FAIL'}"
        )
        print(f"   Std ≈ 1 check: {'✅ PASS' if max_std_dev < 0.01 else '❌ FAIL'}")

        self.verification_results["normality"] = verify_df
        return verify_df

    def export_verification_csv(self, sample_date, output_dir="verification"):
        """
        Export a single date's data + stats to CSV for manual Excel verification.
        """
        os.makedirs(output_dir, exist_ok=True)

        if isinstance(sample_date, str):
            sample_date = pd.Timestamp(sample_date)

        # Get data for date
        date_df = self.df.loc[sample_date].reset_index()
        if "Date" in date_df.columns:
            date_df = date_df.drop("Date", axis=1)

        # Calculate stats
        stats_rows = []
        for stat_name in ["MEAN", "STD", "COUNT", "MIN", "MAX"]:
            row = {"Date": sample_date, "Ticker": f"__{stat_name}__"}
            for col in self.numeric_cols:
                values = date_df[col].dropna()
                if len(values) > 0:
                    if stat_name == "MEAN":
                        row[col] = values.mean()
                    elif stat_name == "STD":
                        row[col] = values.std(ddof=1)
                    elif stat_name == "COUNT":
                        row[col] = len(values)
                    elif stat_name == "MIN":
                        row[col] = values.min()
                    elif stat_name == "MAX":
                        row[col] = values.max()
            stats_rows.append(row)

        stats_df = pd.DataFrame(stats_rows)

        # Combine
        combined = pd.concat([date_df, stats_df], ignore_index=True)

        # Save
        output_file = f"{output_dir}/verify_{sample_date.strftime('%Y%m%d')}.csv"
        combined.to_csv(output_file, index=False)

        print(f"\n💾 Verification CSV saved: {output_file}")
        print(f"   Rows: {len(date_df)} tickers + 5 stat rows")

        # Excel hint
        sample_col = self.numeric_cols[0]
        print(f"\n📊 Excel Verification Formula:")
        print(
            f"   =({sample_col}2 - {sample_col}{len(date_df)+2}) / {sample_col}{len(date_df)+3}"
        )
        print(
            f"   (Row 2 = first ticker, row {len(date_df)+2} = mean, {len(date_df)+3} = std)"
        )

        return output_file

    def detect_anomalies(self, zscore_limit=3.5):
        """Detect extreme Z-scores."""
        print(f"\n🔍 Checking for anomalies (|zscore| > {zscore_limit})...")
        anomalies = {}

        for col in self.numeric_cols:
            col_data = self.df[col]
            extreme = col_data[col_data.abs() > zscore_limit].dropna()

            if len(extreme) > 0:
                anomalies[col] = {"count": len(extreme), "max_abs": extreme.abs().max()}

        if anomalies:
            print(f"   ⚠️  Found anomalies in {len(anomalies)} columns:")
            for col, info in list(anomalies.items())[:5]:  # Show first 5
                print(
                    f"      {col}: {info['count']} cases, max|z|={info['max_abs']:.2f}"
                )
        else:
            print(f"   ✅ No extreme anomalies detected")

        return anomalies


# Helper functions
def quick_verify(final_cache_df, sample_date=None):
    """One-liner verification."""
    verifier = PostBuildVerifier(final_cache_df)

    # 1. Summary
    verifier.generate_summary_report()

    # 2. Auto-pick date if not provided
    if sample_date is None:
        sample_date = verifier.available_dates[
            len(verifier.available_dates) // 2
        ]  # Middle date

    print(f"\n{'='*60}")
    verifier.verify_cross_sectional_normality(sample_dates=[pd.Timestamp(sample_date)])

    # 3. Export CSV
    verifier.export_verification_csv(pd.Timestamp(sample_date))

    # 4. Anomalies
    verifier.detect_anomalies()

    return verifier


def deep_audit(final_cache_df, output_folder="deep_audit"):
    """Comprehensive audit."""
    verifier = PostBuildVerifier(final_cache_df)
    verifier.generate_summary_report()

    # Verify 20 random dates - convert to pandas Timestamps
    sample_indices = np.random.choice(
        len(verifier.available_dates),
        min(20, len(verifier.available_dates)),
        replace=False,
    )
    sample_dates = [
        pd.Timestamp(verifier.available_dates[i]) for i in sample_indices
    ]  # Convert to Timestamp

    verifier.verify_cross_sectional_normality(sample_dates=sample_dates)

    # Export 5 dates
    for date in sample_dates[:5]:
        verifier.export_verification_csv(date, output_dir=output_folder)

    verifier.detect_anomalies()
    return verifier


#

In [39]:
# Method 1: Auto-detect valid dates (safest)
verifier = PostBuildVerifier(final_cache_df)
verifier.verify_cross_sectional_normality()  # Auto-picks 5 dates

# Method 2: Use specific date from 2025
verifier.verify_cross_sectional_normality(sample_dates=["2025-01-15"])

# Method 3: Check what's available first
print(
    f"\nAvailable: {final_cache_df.index.get_level_values('Date').min()} to {final_cache_df.index.get_level_values('Date').max()}"
)


🔍 Verifying 5 sample dates...
   Max |mean| deviation: 6.76e-16 (tolerance: 1e-10)
   Max |std-1| deviation: 3.77e-15
   Mean ≈ 0 check: ✅ PASS
   Std ≈ 1 check: ✅ PASS

🔍 Verifying 1 sample dates...
   Max |mean| deviation: 5.63e-16 (tolerance: 1e-10)
   Max |std-1| deviation: 6.88e-15
   Mean ≈ 0 check: ✅ PASS
   Std ≈ 1 check: ✅ PASS

Available: 2025-01-02 00:00:00 to 2026-03-24 00:00:00


In [40]:
# Check available dates first
available_dates = final_cache_df.index.get_level_values("Date").unique()
print(f"First date: {available_dates.min()}")
print(f"Last date: {available_dates.max()}")
print(f"Total dates: {len(available_dates)}")

# Use a date that actually exists (e.g., first available date)
sample_date = available_dates[0]  # or "2025-01-02", etc.
verifier = quick_verify(final_cache_df, sample_date=sample_date)

First date: 2025-01-02 00:00:00
Last date: 2026-03-24 00:00:00
Total dates: 306

FINAL CACHE VERIFICATION REPORT

📐 Dataset Shape: (285767, 26)
📅 Date Range: 2025-01-02 00:00:00 to 2026-03-24 00:00:00
🏷️  Unique Tickers: 1037
📊 Numeric Features: 26

🌍 Global Statistics (all Z-scores pooled):
   Mean: 0.000000 (should be ~0)
   Std:  0.999465 (should be ~1)
   Min:  -30.6757
   Max:  14.5371
   NaN%: 17.31%


🔍 Verifying 1 sample dates...
   Max |mean| deviation: 3.11e-16 (tolerance: 1e-10)
   Max |std-1| deviation: 1.44e-15
   Mean ≈ 0 check: ✅ PASS
   Std ≈ 1 check: ✅ PASS

💾 Verification CSV saved: verification/verify_20250102.csv
   Rows: 926 tickers + 5 stat rows

📊 Excel Verification Formula:
   =(10d_Price_Gain2 - 10d_Price_Gain928) / 10d_Price_Gain929
   (Row 2 = first ticker, row 928 = mean, 929 = std)

🔍 Checking for anomalies (|zscore| > 3.5)...
   ⚠️  Found anomalies in 24 columns:
      10d_Price_Gain: 2675 cases, max|z|=30.14
      10d_Sharpe: 1402 cases, max|z|=10.77
    

In [41]:
deep_audit(final_cache_df)


FINAL CACHE VERIFICATION REPORT

📐 Dataset Shape: (285767, 26)
📅 Date Range: 2025-01-02 00:00:00 to 2026-03-24 00:00:00
🏷️  Unique Tickers: 1037
📊 Numeric Features: 26

🌍 Global Statistics (all Z-scores pooled):
   Mean: 0.000000 (should be ~0)
   Std:  0.999465 (should be ~1)
   Min:  -30.6757
   Max:  14.5371
   NaN%: 17.31%

🔍 Verifying 20 sample dates...
   Max |mean| deviation: 7.44e-16 (tolerance: 1e-10)
   Max |std-1| deviation: 9.33e-15
   Mean ≈ 0 check: ✅ PASS
   Std ≈ 1 check: ✅ PASS

💾 Verification CSV saved: deep_audit/verify_20250311.csv
   Rows: 927 tickers + 5 stat rows

📊 Excel Verification Formula:
   =(10d_Price_Gain2 - 10d_Price_Gain929) / 10d_Price_Gain930
   (Row 2 = first ticker, row 929 = mean, 930 = std)

💾 Verification CSV saved: deep_audit/verify_20260220.csv
   Rows: 941 tickers + 5 stat rows

📊 Excel Verification Formula:
   =(10d_Price_Gain2 - 10d_Price_Gain943) / 10d_Price_Gain944
   (Row 2 = first ticker, row 943 = mean, 944 = std)

💾 Verification CSV s

In [43]:
sample_date

Timestamp('2025-01-02 00:00:00')

In [51]:
# # After your build finishes:
# final_cache_df = FeatureCubeStitcher.assemble(CHECKPOINT_DIR, FINAL_FILE)

# ============ OPTION 1: Quick 30-Second Check ============
verifier = quick_verify(final_cache_df, sample_date="2026-03-23")

# ============ OPTION 2: Deep Audit (100 dates to CSV) ============
verifier = deep_audit(final_cache_df, output_folder="my_audit")

# ============ OPTION 3: Manual Step-by-Step ============
verifier = PostBuildVerifier(final_cache_df)

# Just summary
verifier.generate_summary_report()

# Verify specific dates
verifier.verify_cross_sectional_normality(
    sample_dates=["2026-01-05", "2026-02-02", "2026-03-02"]
)

# Export one date to check in Excel
verifier.export_verification_csv("2026-03-16")

# Check for data quality issues
verifier.detect_anomalies(zscore_limit=3.5)

# # Full export for Excel power analysis
# verifier.export_full_audit(output_dir="excel_audit", max_dates=50)


FINAL CACHE VERIFICATION REPORT

📐 Dataset Shape: (285767, 26)
📅 Date Range: 2025-01-02 00:00:00 to 2026-03-24 00:00:00
🏷️  Unique Tickers: 1037
📊 Numeric Features: 26

🌍 Global Statistics (all Z-scores pooled):
   Mean: 0.000000 (should be ~0)
   Std:  0.999465 (should be ~1)
   Min:  -30.6757
   Max:  14.5371
   NaN%: 17.31%


🔍 Verifying 1 sample dates...
   Max |mean| deviation: 2.77e-16 (tolerance: 1e-10)
   Max |std-1| deviation: 2.44e-15
   Mean ≈ 0 check: ✅ PASS
   Std ≈ 1 check: ✅ PASS

💾 Verification CSV saved: verification/verify_20260323.csv
   Rows: 943 tickers + 5 stat rows

📊 Excel Verification Formula:
   =(10d_Price_Gain2 - 10d_Price_Gain945) / 10d_Price_Gain946
   (Row 2 = first ticker, row 945 = mean, 946 = std)

🔍 Checking for anomalies (|zscore| > 3.5)...
   ⚠️  Found anomalies in 24 columns:
      10d_Price_Gain: 2675 cases, max|z|=30.14
      10d_Sharpe: 1402 cases, max|z|=10.77
      10d_Sharpe_ATRP: 1269 cases, max|z|=7.61
      10d_Sharpe_TRP: 751 cases, max|

{'10d_Price_Gain': {'count': 2675, 'max_abs': 30.13548373003792},
 '10d_Sharpe': {'count': 1402, 'max_abs': 10.766793065765484},
 '10d_Sharpe_ATRP': {'count': 1269, 'max_abs': 7.614733434423297},
 '10d_Sharpe_TRP': {'count': 751, 'max_abs': 30.67574058511653},
 '10d_VIX_Filtered_Momentum_4': {'count': 2302, 'max_abs': 14.537148667885386},
 '10d_Info_Ratio_Stdev_Alpha_63d': {'count': 105,
  'max_abs': 4.777886500541887},
 '10d_Consistency_WinRate_5d': {'count': 24, 'max_abs': 4.62826488891645},
 '10d_Oversold_RSI': {'count': 1240, 'max_abs': 7.666311146184375},
 '10d_Dip_Buyer_Drawdown_dd_21': {'count': 3159,
  'max_abs': 13.439278146121097},
 '10d_Low_Volatility_ATRP': {'count': 3049, 'max_abs': 9.006915824327502},
 '10d_VIX_Filtered_Momentum_10': {'count': 2302,
  'max_abs': 14.537148667885386},
 '21d_Price_Gain': {'count': 2534, 'max_abs': 29.25990338611189},
 '21d_Sharpe': {'count': 1807, 'max_abs': 10.858241831788463},
 '21d_Sharpe_ATRP': {'count': 1673, 'max_abs': 8.84217144265746

In [ ]:
import pandas as pd


def compare_ticker_universes(df_ohlcv, final_cache_df, verification_date):
    """
    Compare tickers in df_ohlcv vs final_cache_df for a specific date.
    """

    verif_date = pd.Timestamp(verification_date)

    # Get tickers from final_cache_df for this date
    try:
        cache_slice = final_cache_df.loc[verif_date]
        cache_tickers = set(cache_slice.index.tolist())
        print(f"✅ final_cache_df date found: {verif_date.date()}")
    except KeyError:
        # Find nearest
        all_cache_dates = final_cache_df.index.get_level_values("Date").unique()
        nearest = all_cache_dates[
            all_cache_dates.get_indexer([verif_date], method="nearest")[0]
        ]
        cache_slice = final_cache_df.loc[nearest]
        cache_tickers = set(cache_slice.index.tolist())
        print(f"⚠️  Using nearest cache date: {nearest.date()}")
        verif_date = nearest

    # Get tickers from df_ohlcv with data on this date AND 10 days prior
    all_ohlcv_dates = df_ohlcv.index.get_level_values("Date").unique().sort_values()

    # Find exact or nearest date
    if verif_date not in all_ohlcv_dates:
        nearest_idx = all_ohlcv_dates.get_indexer([verif_date], method="nearest")[0]
        ohlcv_end_date = all_ohlcv_dates[nearest_idx]
    else:
        ohlcv_end_date = verif_date

    # Get start date (10 days back)
    end_idx = (
        all_ohlcv_dates.get_loc(ohlcv_end_date)
        if ohlcv_end_date in all_ohlcv_dates
        else all_ohlcv_dates.get_indexer([ohlcv_end_date], method="nearest")[0]
    )
    start_idx = end_idx - 10
    start_date = all_ohlcv_dates[start_idx] if start_idx >= 0 else all_ohlcv_dates[0]

    # Tickers with data on BOTH dates
    idx = pd.IndexSlice
    tickers_end = set(
        df_ohlcv.loc[idx[:, ohlcv_end_date], :]
        .index.get_level_values("Ticker")
        .unique()
    )
    tickers_start = set(
        df_ohlcv.loc[idx[:, start_date], :].index.get_level_values("Ticker").unique()
    )
    ohlcv_tickers = tickers_start.intersection(tickers_end)

    print(f"\n{'='*70}")
    print(f"TICKER UNIVERSE COMPARISON for {verif_date.date()}")
    print(f"{'='*70}")

    print(f"\n📊 final_cache_df:")
    print(f"   Total tickers: {len(cache_tickers)}")
    print(f"   Sample: {sorted(list(cache_tickers))[:5]}")

    print(
        f"\n📊 df_ohlcv (with data on {start_date.date()} and {ohlcv_end_date.date()}):"
    )
    print(f"   Total tickers: {len(ohlcv_tickers)}")
    print(f"   Sample: {sorted(list(ohlcv_tickers))[:5]}")

    # Compare
    common = cache_tickers.intersection(ohlcv_tickers)
    only_cache = cache_tickers - ohlcv_tickers
    only_ohlcv = ohlcv_tickers - cache_tickers

    print(f"\n🔍 Comparison:")
    print(f"   Common tickers: {len(common)}")
    print(f"   Only in cache: {len(only_cache)}")
    print(f"   Only in ohlcv: {len(only_ohlcv)}")

    if only_cache:
        print(f"\n   Tickers in cache but not ohlcv: {sorted(list(only_cache))[:10]}")
    if only_ohlcv:
        print(f"\n   Tickers in ohlcv but not cache: {sorted(list(only_ohlcv))[:10]}")

    # Check if 'A' is in both
    print(f"\n🔍 Ticker 'A' check:")
    print(f"   In cache: {'A' in cache_tickers}")
    print(f"   In ohlcv: {'A' in ohlcv_tickers}")
    if "A" in ohlcv_tickers:
        try:
            p_start = df_ohlcv.loc[idx["A", start_date], "Adj Close"]
            p_end = df_ohlcv.loc[idx["A", ohlcv_end_date], "Adj Close"]
            print(f"   A start price ({start_date.date()}): ${p_start:.4f}")
            print(f"   A end price ({ohlcv_end_date.date()}): ${p_end:.4f}")
        except:
            print(f"   ⚠️  Could not get A prices")

    return {
        "cache_tickers": cache_tickers,
        "ohlcv_tickers": ohlcv_tickers,
        "common": common,
        "only_cache": only_cache,
        "only_ohlcv": only_ohlcv,
        "ohlcv_end_date": ohlcv_end_date,
        "ohlcv_start_date": start_date,
    }


def check_specific_ticker(
    df_ohlcv, final_cache_df, ticker, verification_date, lookback=10
):
    """
    Detailed check for a specific ticker across both data sources.
    """

    verif_date = pd.Timestamp(verification_date)
    result = compare_ticker_universes(df_ohlcv, final_cache_df, verif_date)

    print(f"\n{'='*70}")
    print(f"DETAILED CHECK FOR TICKER '{ticker}'")
    print(f"{'='*70}")

    # Check cache
    try:
        cache_z = final_cache_df.loc[
            (result.get("cache_date", verif_date), ticker), f"{lookback}d_Price_Gain"
        ]
        print(f"\n✅ Cache: Z-score = {cache_z:.6f}")
    except KeyError:
        print(f"\n❌ Cache: Ticker '{ticker}' not found")
        cache_z = None

    # Check ohlcv
    if ticker in result["ohlcv_tickers"]:
        idx = pd.IndexSlice
        p_start = df_ohlcv.loc[idx[ticker, result["ohlcv_start_date"]], "Adj Close"]
        p_end = df_ohlcv.loc[idx[ticker, result["ohlcv_end_date"]], "Adj Close"]

        # Handle series
        if isinstance(p_start, pd.Series):
            p_start = p_start.iloc[0]
        if isinstance(p_end, pd.Series):
            p_end = p_end.iloc[0]

        simple_ret = (p_end / p_start) - 1
        log_ret = np.log(p_end / p_start)

        print(f"\n✅ OHLCV:")
        print(f"   Start ({result['ohlcv_start_date'].date()}): ${p_start:.4f}")
        print(f"   End ({result['ohlcv_end_date'].date()}): ${p_end:.4f}")
        print(f"   Simple return: {simple_ret:.6f}")
        print(f"   Log return: {log_ret:.6f}")

        # Calculate what Z-score would be with different universes
        if cache_z is not None:
            print(f"\n🔍 Z-score reconciliation:")
            print(f"   Cache Z: {cache_z:.6f}")
            print(f"   Your log return: {log_ret:.6f}")

            # What cross-sectional stats would produce this Z?
            # Z = (X - mean) / std  =>  mean = X - Z*std
            # We need to find what universe produces matching stats

    else:
        print(f"\n❌ OHLCV: Ticker '{ticker}' not found in date range")

    return result


#

In [80]:
# # Compare universes
# universe_comparison = compare_ticker_universes(df_ohlcv, final_cache_df, "2025-01-02")

# Detailed check for ticker A
detailed_check = check_specific_ticker(
    df_ohlcv, final_cache_df, "A", "2025-01-02", lookback=10
)

✅ final_cache_df date found: 2025-01-02

TICKER UNIVERSE COMPARISON for 2025-01-02

📊 final_cache_df:
   Total tickers: 926
   Sample: ['A', 'AA', 'AAL', 'AAPL', 'ABBV']

📊 df_ohlcv (with data on 2024-12-17 and 2025-01-02):
   Total tickers: 1555
   Sample: ['A', 'AA', 'AAL', 'AAON', 'AAPL']

🔍 Comparison:
   Common tickers: 926
   Only in cache: 0
   Only in ohlcv: 629

   Tickers in ohlcv but not cache: ['AAON', 'ABEV', 'ABVX', 'ACI', 'ACWX', 'ADC', 'ADT', 'AEG', 'AEIS', 'AFG']

🔍 Ticker 'A' check:
   In cache: True
   In ohlcv: True
   A start price (2024-12-17): $135.1210
   A end price (2025-01-02): $132.3640

DETAILED CHECK FOR TICKER 'A'

✅ Cache: Z-score = 0.235365

✅ OHLCV:
   Start (2024-12-17): $135.1210
   End (2025-01-02): $132.3640
   Simple return: -0.020404
   Log return: -0.020615

🔍 Z-score reconciliation:
   Cache Z: 0.235365
   Your log return: -0.020615


In [ ]:
import pandas as pd
import numpy as np


def verify_zscore_calculation(df_ohlcv, final_cache_df, verification_date, lookback=10):
    """
    Verify Z-score calculation using ONLY common tickers between both datasets.
    """

    verif_date = pd.Timestamp(verification_date)

    # Get all dates
    all_dates = df_ohlcv.index.get_level_values("Date").unique().sort_values()
    cache_dates = final_cache_df.index.get_level_values("Date").unique()

    # Find exact or nearest date in both datasets
    if verif_date not in all_dates:
        nearest_idx = all_dates.get_indexer([verif_date], method="nearest")[0]
        actual_date = all_dates[nearest_idx]
        print(f"⚠️  Using nearest ohlcv date: {actual_date.date()}")
    else:
        actual_date = verif_date

    if actual_date not in cache_dates:
        nearest_idx = cache_dates.get_indexer([actual_date], method="nearest")[0]
        cache_date = cache_dates[nearest_idx]
        print(f"⚠️  Using nearest cache date: {cache_date.date()}")
    else:
        cache_date = actual_date

    # Calculate lookback window
    date_list = all_dates.tolist()
    end_idx = (
        date_list.index(actual_date)
        if actual_date in date_list
        else [
            i
            for i, d in enumerate(date_list)
            if str(d.date()) == str(actual_date.date())
        ][0]
    )

    start_idx = end_idx - lookback
    if start_idx < 0:
        raise ValueError(f"Not enough history for {lookback}d calculation")
    start_date = date_list[start_idx]

    print(f"\n{'='*70}")
    print(f"Z-SCORE VERIFICATION: {lookback}d_Price_Gain")
    print(f"Date: {cache_date.date()} (cache) / {actual_date.date()} (ohlcv)")
    print(f"Lookback: {start_date.date()} to {actual_date.date()}")
    print(f"{'='*70}")

    # === KEY FIX: Get ONLY common tickers ===
    idx = pd.IndexSlice

    # Tickers from cache for this date
    cache_slice = final_cache_df.loc[cache_date]
    cache_tickers = set(cache_slice.index.tolist())
    print(f"\n📊 Cache tickers: {len(cache_tickers)}")

    # Tickers from ohlcv with data on BOTH dates
    tickers_end = set(
        df_ohlcv.loc[idx[:, actual_date], :].index.get_level_values("Ticker").unique()
    )
    tickers_start = set(
        df_ohlcv.loc[idx[:, start_date], :].index.get_level_values("Ticker").unique()
    )
    ohlcv_tickers = tickers_start.intersection(tickers_end)
    print(f"📊 OHLCV tickers (with data on both dates): {len(ohlcv_tickers)}")

    # === USE ONLY COMMON TICKERS ===
    common_tickers = cache_tickers.intersection(ohlcv_tickers)
    print(f"✅ Common tickers: {len(common_tickers)}")

    # Step 1: Calculate LOG RETURNS for COMMON tickers only
    print(
        f"\n📊 Step 1: Calculating log returns for {len(common_tickers)} common tickers..."
    )

    log_returns = []
    for ticker in common_tickers:
        try:
            p0 = df_ohlcv.loc[idx[ticker, start_date], "Adj Close"]
            p_end = df_ohlcv.loc[idx[ticker, actual_date], "Adj Close"]

            if isinstance(p0, pd.Series):
                p0 = p0.iloc[0]
            if isinstance(p_end, pd.Series):
                p_end = p_end.iloc[0]

            log_return = np.log(p_end / p0)
            log_returns.append(
                {
                    "Ticker": ticker,
                    f"{lookback}d_Log_Return": log_return,
                    "Cache_ZScore": cache_slice.loc[ticker, f"{lookback}d_Price_Gain"],
                }
            )

        except (KeyError, IndexError):
            continue

    # Create DataFrame with all data
    df = pd.DataFrame(log_returns).set_index("Ticker")
    print(f"   Successfully calculated: {len(df)} tickers")

    # Step 2: Calculate cross-sectional Z-scores
    print(f"\n📊 Step 2: Calculating cross-sectional statistics...")

    mean_log = df[f"{lookback}d_Log_Return"].mean()
    std_log = df[f"{lookback}d_Log_Return"].std(ddof=1)

    print(f"   Mean log return: {mean_log:.6f}")
    print(f"   Std log return:  {std_log:.6f} (ddof=1)")

    # Calculate Z-scores
    df["Calculated_ZScore"] = (df[f"{lookback}d_Log_Return"] - mean_log) / std_log

    # Step 3: Compare
    print(f"\n📊 Step 3: Comparing with cache...")

    df["Difference"] = df["Calculated_ZScore"] - df["Cache_ZScore"]
    df["Abs_Difference"] = df["Difference"].abs()

    max_diff = df["Abs_Difference"].max()
    mean_diff = df["Abs_Difference"].mean()
    exact_matches = (df["Abs_Difference"] < 1e-10).sum()

    print(f"\n{'='*70}")
    print(f"✅ VERIFICATION RESULTS")
    print(f"{'='*70}")
    print(f"Tickers compared: {len(df)}")
    print(
        f"Exact matches (< 1e-10): {exact_matches} ({100*exact_matches/len(df):.1f}%)"
    )
    print(f"Max absolute difference: {max_diff:.2e}")
    print(f"Mean absolute difference: {mean_diff:.2e}")

    # Show ticker A specifically
    if "A" in df.index:
        row = df.loc["A"]
        print(f"\n{'='*70}")
        print(f"TICKER 'A' DETAIL")
        print(f"{'='*70}")
        print(f"  Log Return:     {row[f'{lookback}d_Log_Return']:.6f}")
        print(f"  Cross-sec Mean: {mean_log:.6f}")
        print(f"  Cross-sec Std:  {std_log:.6f}")
        print(f"  Calculated Z:   {row['Calculated_ZScore']:.6f}")
        print(f"  Cache Z:        {row['Cache_ZScore']:.6f}")
        print(f"  Difference:     {row['Difference']:.2e}")
        print(f"  Match:          {'✅' if row['Abs_Difference'] < 1e-10 else '❌'}")

    # Show worst mismatches if any
    if max_diff > 1e-6:
        print(f"\n⚠️  Worst mismatches:")
        print(
            df.nlargest(3, "Abs_Difference")[
                ["Calculated_ZScore", "Cache_ZScore", "Difference"]
            ]
        )

    return df

In [82]:
# Now uses ONLY common tickers - should match perfectly
results = verify_zscore_calculation(df_ohlcv, final_cache_df, "2025-01-02", lookback=10)


Z-SCORE VERIFICATION: 10d_Price_Gain
Date: 2025-01-02 (cache) / 2025-01-02 (ohlcv)
Lookback: 2024-12-17 to 2025-01-02

📊 Cache tickers: 926
📊 OHLCV tickers (with data on both dates): 1555
✅ Common tickers: 926

📊 Step 1: Calculating log returns for 926 common tickers...
   Successfully calculated: 926 tickers

📊 Step 2: Calculating cross-sectional statistics...
   Mean log return: -0.031414
   Std log return:  0.045883 (ddof=1)

📊 Step 3: Comparing with cache...

✅ VERIFICATION RESULTS
Tickers compared: 926
Exact matches (< 1e-10): 926 (100.0%)
Max absolute difference: 1.78e-15
Mean absolute difference: 1.99e-16

TICKER 'A' DETAIL
  Log Return:     -0.020615
  Cross-sec Mean: -0.031414
  Cross-sec Std:  0.045883
  Calculated Z:   0.235365
  Cache Z:        0.235365
  Difference:     -5.55e-17
  Match:          ✅


In [ ]:
import pandas as pd
import numpy as np


def verify_zscore_calculation(
    df_ohlcv,
    final_cache_df,
    verification_date,
    lookback=10,
    extreme_threshold=2.0,
    top_n=10,
):
    """
    Verify Z-score calculation and identify exceptional performers.

    Parameters:
    -----------
    extreme_threshold : float
        Z-score threshold for "exceptional" (default: 2.0 = top/bottom 2.5%)
    top_n : int
        Number of top/bottom tickers to display
    """

    verif_date = pd.Timestamp(verification_date)

    # Get all dates
    all_dates = df_ohlcv.index.get_level_values("Date").unique().sort_values()
    cache_dates = final_cache_df.index.get_level_values("Date").unique()

    # Find exact or nearest date in both datasets
    if verif_date not in all_dates:
        nearest_idx = all_dates.get_indexer([verif_date], method="nearest")[0]
        actual_date = all_dates[nearest_idx]
        print(f"⚠️  Using nearest ohlcv date: {actual_date.date()}")
    else:
        actual_date = verif_date

    if actual_date not in cache_dates:
        nearest_idx = cache_dates.get_indexer([actual_date], method="nearest")[0]
        cache_date = cache_dates[nearest_idx]
        print(f"⚠️  Using nearest cache date: {cache_date.date()}")
    else:
        cache_date = actual_date

    # Calculate lookback window
    date_list = all_dates.tolist()
    end_idx = (
        date_list.index(actual_date)
        if actual_date in date_list
        else [
            i
            for i, d in enumerate(date_list)
            if str(d.date()) == str(actual_date.date())
        ][0]
    )

    start_idx = end_idx - lookback
    if start_idx < 0:
        raise ValueError(f"Not enough history for {lookback}d calculation")
    start_date = date_list[start_idx]

    print(f"\n{'='*70}")
    print(f"Z-SCORE VERIFICATION: {lookback}d_Price_Gain")
    print(f"Date: {cache_date.date()} (cache) / {actual_date.date()} (ohlcv)")
    print(f"Lookback: {start_date.date()} to {actual_date.date()}")
    print(f"{'='*70}")

    # Get ONLY common tickers
    idx = pd.IndexSlice

    cache_slice = final_cache_df.loc[cache_date]
    cache_tickers = set(cache_slice.index.tolist())
    print(f"\n📊 Cache tickers: {len(cache_tickers)}")

    tickers_end = set(
        df_ohlcv.loc[idx[:, actual_date], :].index.get_level_values("Ticker").unique()
    )
    tickers_start = set(
        df_ohlcv.loc[idx[:, start_date], :].index.get_level_values("Ticker").unique()
    )
    ohlcv_tickers = tickers_start.intersection(tickers_end)
    print(f"📊 OHLCV tickers (with data on both dates): {len(ohlcv_tickers)}")

    common_tickers = cache_tickers.intersection(ohlcv_tickers)
    print(f"✅ Common tickers: {len(common_tickers)}")

    # Calculate LOG RETURNS for COMMON tickers
    print(
        f"\n📊 Step 1: Calculating log returns for {len(common_tickers)} common tickers..."
    )

    log_returns = []
    for ticker in common_tickers:
        try:
            p0 = df_ohlcv.loc[idx[ticker, start_date], "Adj Close"]
            p_end = df_ohlcv.loc[idx[ticker, actual_date], "Adj Close"]

            if isinstance(p0, pd.Series):
                p0 = p0.iloc[0]
            if isinstance(p_end, pd.Series):
                p_end = p_end.iloc[0]

            log_return = np.log(p_end / p0)
            log_returns.append(
                {
                    "Ticker": ticker,
                    f"{lookback}d_Log_Return": log_return,
                    "Cache_ZScore": cache_slice.loc[ticker, f"{lookback}d_Price_Gain"],
                }
            )

        except (KeyError, IndexError):
            continue

    df = pd.DataFrame(log_returns).set_index("Ticker")
    print(f"   Successfully calculated: {len(df)} tickers")

    # Calculate cross-sectional Z-scores
    print(f"\n📊 Step 2: Calculating cross-sectional statistics...")

    mean_log = df[f"{lookback}d_Log_Return"].mean()
    std_log = df[f"{lookback}d_Log_Return"].std(ddof=1)

    print(f"   Mean log return: {mean_log:.6f}")
    print(f"   Std log return:  {std_log:.6f} (ddof=1)")

    df["Calculated_ZScore"] = (df[f"{lookback}d_Log_Return"] - mean_log) / std_log

    # Compare with cache
    print(f"\n📊 Step 3: Comparing with cache...")

    df["Difference"] = df["Calculated_ZScore"] - df["Cache_ZScore"]
    df["Abs_Difference"] = df["Difference"].abs()

    max_diff = df["Abs_Difference"].max()
    mean_diff = df["Abs_Difference"].mean()
    exact_matches = (df["Abs_Difference"] < 1e-10).sum()

    print(f"\n{'='*70}")
    print(f"✅ VERIFICATION RESULTS")
    print(f"{'='*70}")
    print(f"Tickers compared: {len(df)}")
    print(
        f"Exact matches (< 1e-10): {exact_matches} ({100*exact_matches/len(df):.1f}%)"
    )
    print(f"Max absolute difference: {max_diff:.2e}")
    print(f"Mean absolute difference: {mean_diff:.2e}")

    # ============ EXCEPTIONAL Z-SCORE ANALYSIS ============
    print(f"\n{'='*70}")
    print(f"🌟 EXCEPTIONAL Z-SCORE ANALYSIS (|Z| > {extreme_threshold})")
    print(f"{'='*70}")

    # Add rank and percentile
    df["Z_Rank"] = df["Cache_ZScore"].rank(ascending=False)
    df["Z_Percentile"] = df["Cache_ZScore"].rank(pct=True) * 100

    # Extreme positive
    extreme_high = df[df["Cache_ZScore"] > extreme_threshold].sort_values(
        "Cache_ZScore", ascending=False
    )
    print(f"\n🔥 TOP PERFORMERS (Z > {extreme_threshold}):")
    print(
        f"   Count: {len(extreme_high)} tickers ({100*len(extreme_high)/len(df):.1f}% of universe)"
    )

    if len(extreme_high) > 0:
        display_cols = [
            "Cache_ZScore",
            "Z_Rank",
            "Z_Percentile",
            f"{lookback}d_Log_Return",
        ]
        print(f"\n   Top {min(top_n, len(extreme_high))} by Z-score:")
        print(extreme_high[display_cols].head(top_n).to_string())

        # Summary stats
        print(f"\n   Extreme high stats:")
        print(f"      Mean Z: {extreme_high['Cache_ZScore'].mean():.3f}")
        print(f"      Max Z:  {extreme_high['Cache_ZScore'].max():.3f}")
        print(
            f"      Mean log return: {extreme_high[f'{lookback}d_Log_Return'].mean():.4f}"
        )

    # Extreme negative
    extreme_low = df[df["Cache_ZScore"] < -extreme_threshold].sort_values(
        "Cache_ZScore", ascending=True
    )
    print(f"\n❄️  BOTTOM PERFORMERS (Z < -{extreme_threshold}):")
    print(
        f"   Count: {len(extreme_low)} tickers ({100*len(extreme_low)/len(df):.1f}% of universe)"
    )

    if len(extreme_low) > 0:
        display_cols = [
            "Cache_ZScore",
            "Z_Rank",
            "Z_Percentile",
            f"{lookback}d_Log_Return",
        ]
        print(f"\n   Bottom {min(top_n, len(extreme_low))} by Z-score:")
        print(extreme_low[display_cols].head(top_n).to_string())

        print(f"\n   Extreme low stats:")
        print(f"      Mean Z: {extreme_low['Cache_ZScore'].mean():.3f}")
        print(f"      Min Z:  {extreme_low['Cache_ZScore'].min():.3f}")
        print(
            f"      Mean log return: {extreme_low[f'{lookback}d_Log_Return'].mean():.4f}"
        )

    # Distribution summary
    print(f"\n📊 Z-SCORE DISTRIBUTION:")
    print(f"   Mean: {df['Cache_ZScore'].mean():.6f}")
    print(f"   Std:  {df['Cache_ZScore'].std(ddof=1):.6f}")
    print(f"   Min:  {df['Cache_ZScore'].min():.3f}")
    print(f"   Max:  {df['Cache_ZScore'].max():.3f}")
    print(f"   |Z| > 2: {len(df[df['Cache_ZScore'].abs() > 2])} tickers")
    print(f"   |Z| > 3: {len(df[df['Cache_ZScore'].abs() > 3])} tickers")

    # Ticker A detail
    if "A" in df.index:
        row = df.loc["A"]
        print(f"\n{'='*70}")
        print(f"TICKER 'A' DETAIL")
        print(f"{'='*70}")
        print(f"  Log Return:     {row[f'{lookback}d_Log_Return']:.6f}")
        print(f"  Cross-sec Mean: {mean_log:.6f}")
        print(f"  Cross-sec Std:  {std_log:.6f}")
        print(f"  Calculated Z:   {row['Calculated_ZScore']:.6f}")
        print(f"  Cache Z:        {row['Cache_ZScore']:.6f}")
        print(
            f"  Z-Percentile:   {row['Z_Percentile']:.1f}% (Rank #{int(row['Z_Rank'])} of {len(df)})"
        )
        print(f"  Difference:     {row['Difference']:.2e}")
        print(f"  Match:          {'✅' if row['Abs_Difference'] < 1e-10 else '❌'}")

    return df


#

In [76]:
# Default: show tickers with |Z| > 2.0 (top/bottom 2.5%)
results = verify_zscore_calculation(df_ohlcv, final_cache_df, "2025-01-02", lookback=10)

# Stricter: |Z| > 2.5 (top/bottom 0.6%)
results = verify_zscore_calculation(
    df_ohlcv, final_cache_df, "2025-01-02", lookback=10, extreme_threshold=2.5, top_n=5
)

# Show top 20 instead of 10
results = verify_zscore_calculation(
    df_ohlcv, final_cache_df, "2025-01-02", lookback=10, extreme_threshold=1.5, top_n=20
)


Z-SCORE VERIFICATION: 10d_Price_Gain
Date: 2025-01-02 (cache) / 2025-01-02 (ohlcv)
Lookback: 2024-12-17 to 2025-01-02

📊 Cache tickers: 926
📊 OHLCV tickers (with data on both dates): 1555
✅ Common tickers: 926

📊 Step 1: Calculating log returns for 926 common tickers...
   Successfully calculated: 926 tickers

📊 Step 2: Calculating cross-sectional statistics...
   Mean log return: -0.031414
   Std log return:  0.045883 (ddof=1)

📊 Step 3: Comparing with cache...

✅ VERIFICATION RESULTS
Tickers compared: 926
Exact matches (< 1e-10): 926 (100.0%)
Max absolute difference: 1.78e-15
Mean absolute difference: 1.99e-16

🌟 EXCEPTIONAL Z-SCORE ANALYSIS (|Z| > 2.0)

🔥 TOP PERFORMERS (Z > 2.0):
   Count: 18 tickers (1.9% of universe)

   Top 10 by Z-score:
        Cache_ZScore  Z_Rank  Z_Percentile  10d_Log_Return
Ticker                                                    
DRI           3.2947     1.0      100.0000          0.1198
AR            3.2708     2.0       99.8920          0.1187
NVDL   

In [ ]:
import pandas as pd
import numpy as np


def verify_zscore_calculation(df_ohlcv, final_cache_df, verification_date, lookback=10):
    """
    Verify Z-score calculation using LOG RETURNS (confirmed correct method).
    """

    verif_date = pd.Timestamp(verification_date)

    # Get all dates from ohlcv
    all_dates = df_ohlcv.index.get_level_values("Date").unique().sort_values()

    # Find exact or nearest date
    if verif_date not in all_dates:
        nearest_idx = all_dates.get_indexer([verif_date], method="nearest")[0]
        actual_date = all_dates[nearest_idx]
        print(f"⚠️  Using nearest date: {actual_date.date()}")
    else:
        actual_date = verif_date

    # Calculate lookback window
    date_list = all_dates.tolist()
    end_idx = (
        date_list.index(actual_date)
        if actual_date in date_list
        else [
            i
            for i, d in enumerate(date_list)
            if str(d.date()) == str(actual_date.date())
        ][0]
    )

    start_idx = end_idx - lookback

    if start_idx < 0:
        raise ValueError(f"Not enough history for {lookback}d calculation")

    start_date = date_list[start_idx]

    print(f"\n{'='*70}")
    print(f"Z-SCORE VERIFICATION: {lookback}d_Price_Gain for {actual_date.date()}")
    print(f"Method: LOG RETURN (ln(P_end/P_start))")
    print(f"Lookback: {start_date.date()} to {actual_date.date()}")
    print(f"{'='*70}")

    # Step 1: Calculate LOG RETURNS for ALL tickers
    print(f"\n📊 Step 1: Calculating log returns from df_ohlcv...")

    idx = pd.IndexSlice

    # Get all tickers available on both dates
    tickers_start = (
        df_ohlcv.loc[idx[:, start_date], :].index.get_level_values("Ticker").unique()
    )
    tickers_end = (
        df_ohlcv.loc[idx[:, actual_date], :].index.get_level_values("Ticker").unique()
    )
    all_tickers = tickers_start.intersection(tickers_end)

    print(f"   Tickers with data on both dates: {len(all_tickers)}")

    log_returns = []

    for ticker in all_tickers:
        try:
            # Get prices
            p0 = df_ohlcv.loc[idx[ticker, start_date], "Adj Close"]
            p_end = df_ohlcv.loc[idx[ticker, actual_date], "Adj Close"]

            # Handle both scalar and Series
            if isinstance(p0, pd.Series):
                p0 = p0.iloc[0]
            if isinstance(p_end, pd.Series):
                p_end = p_end.iloc[0]

            # LOG RETURN: ln(P_end / P_start)
            log_return = np.log(p_end / p0)
            log_returns.append(
                {"Ticker": ticker, f"{lookback}d_Log_Return": log_return}
            )

        except (KeyError, IndexError):
            continue

    raw_df = pd.DataFrame(log_returns).set_index("Ticker")
    print(f"   Successfully calculated: {len(raw_df)} log returns")
    print(
        f"   Log return stats: mean={raw_df[f'{lookback}d_Log_Return'].mean():.6f}, "
        f"std={raw_df[f'{lookback}d_Log_Return'].std(ddof=1):.6f}"
    )

    # Step 2: Get Z-scores from final_cache_df
    print(f"\n📊 Step 2: Getting Z-scores from final_cache_df...")

    try:
        cache_slice = final_cache_df.loc[actual_date]
        z_col = f"{lookback}d_Price_Gain"

        if z_col not in cache_slice.columns:
            print(f"❌ Column {z_col} not found")
            return None

        cache_zscores = cache_slice[[z_col]].copy()
        cache_zscores.columns = ["Cache_ZScore"]
        print(f"   Found {len(cache_zscores)} Z-scores in cache")

    except KeyError:
        print(f"❌ Date {actual_date} not found in final_cache_df")
        return None

    # Step 3: Calculate Z-scores from log returns
    print(f"\n📊 Step 3: Calculating Z-scores...")

    mean_log = raw_df[f"{lookback}d_Log_Return"].mean()
    std_log = raw_df[f"{lookback}d_Log_Return"].std(ddof=1)

    print(f"   Cross-sectional mean: {mean_log:.6f}")
    print(f"   Cross-sectional std (ddof=1): {std_log:.6f}")

    # Z = (X - mean) / std
    raw_df["Calculated_ZScore"] = (
        raw_df[f"{lookback}d_Log_Return"] - mean_log
    ) / std_log

    # Step 4: Merge and compare
    print(f"\n📊 Step 4: Comparing calculations...")

    comparison = raw_df.join(cache_zscores, how="inner")
    comparison["Difference"] = (
        comparison["Calculated_ZScore"] - comparison["Cache_ZScore"]
    )
    comparison["Abs_Difference"] = comparison["Difference"].abs()

    # Statistics
    max_diff = comparison["Abs_Difference"].max()
    mean_diff = comparison["Abs_Difference"].mean()
    exact_matches = (comparison["Abs_Difference"] < 1e-10).sum()

    print(f"\n{'='*70}")
    print(f"✅ VERIFICATION RESULTS")
    print(f"{'='*70}")
    print(f"Tickers compared: {len(comparison)}")
    print(
        f"Exact matches (< 1e-10): {exact_matches} ({100*exact_matches/len(comparison):.1f}%)"
    )
    print(f"Max absolute difference: {max_diff:.2e}")
    print(f"Mean absolute difference: {mean_diff:.2e}")

    # Show first 10 for verification
    print(f"\n✅ Sample verification (first 10 tickers):")
    sample = comparison.head(10)[["Cache_ZScore", "Calculated_ZScore", "Difference"]]
    print(sample.to_string())

    # Detailed breakdown for first 3
    print(f"\n{'='*70}")
    print(f"DETAILED BREAKDOWN (First 3 Tickers)")
    print(f"{'='*70}")

    for ticker in comparison.head(3).index:
        row = comparison.loc[ticker]
        print(f"\n{ticker}:")
        print(f"  Log Return:     {row[f'{lookback}d_Log_Return']:.6f}")
        print(f"  Mean (all):     {mean_log:.6f}")
        print(f"  Std (all):      {std_log:.6f}")
        print(
            f"  Calculation:    ({row[f'{lookback}d_Log_Return']:.6f} - {mean_log:.6f}) / {std_log:.6f}"
        )
        print(
            f"                  = {row[f'{lookback}d_Log_Return'] - mean_log:.6f} / {std_log:.6f}"
        )
        print(f"                  = {row['Calculated_ZScore']:.6f}")
        print(f"  Cache Z-Score:  {row['Cache_ZScore']:.6f}")
        print(f"  Match:          {'✅' if row['Abs_Difference'] < 1e-10 else '❌'}")

    return comparison


def export_verification_csv(
    comparison_df, verification_date, lookback=10, output_file=None
):
    """
    Export detailed verification to CSV for Excel checking.
    """
    if output_file is None:
        output_file = f"zscore_verify_{lookback}d_{pd.Timestamp(verification_date).strftime('%Y%m%d')}.csv"

    export_df = comparison_df.copy()

    # Add verification columns
    z_col = f"{lookback}d_Log_Return"
    mean_log = export_df[z_col].mean()
    std_log = export_df[z_col].std(ddof=1)

    export_df["CrossSec_Mean"] = mean_log
    export_df["CrossSec_Std"] = std_log
    export_df["Excel_Formula"] = f"=(B2-{mean_log:.6f})/{std_log:.6f}"

    # Reorder columns
    cols = [
        z_col,
        "CrossSec_Mean",
        "CrossSec_Std",
        "Calculated_ZScore",
        "Cache_ZScore",
        "Difference",
    ]

    export_df[cols].to_csv(output_file)
    print(f"\n💾 Detailed verification saved: {output_file}")
    print(f"   Excel formula: =(LogReturn - {mean_log:.6f}) / {std_log:.6f}")

    return output_file


#

In [62]:
# Verify all 926 tickers using log returns
verification_results = verify_zscore_calculation(
    df_ohlcv=df_ohlcv,
    final_cache_df=final_cache_df,
    verification_date="2025-01-02",
    lookback=10,
)

# Export to CSV
export_verification_csv(verification_results, "2025-01-02", 10)


Z-SCORE VERIFICATION: 10d_Price_Gain for 2025-01-02
Method: LOG RETURN (ln(P_end/P_start))
Lookback: 2024-12-17 to 2025-01-02

📊 Step 1: Calculating log returns from df_ohlcv...
   Tickers with data on both dates: 1555
   Successfully calculated: 1555 log returns
   Log return stats: mean=-0.028852, std=0.046619

📊 Step 2: Getting Z-scores from final_cache_df...
   Found 926 Z-scores in cache

📊 Step 3: Calculating Z-scores...
   Cross-sectional mean: -0.028852
   Cross-sectional std (ddof=1): 0.046619

📊 Step 4: Comparing calculations...

✅ VERIFICATION RESULTS
Tickers compared: 926
Exact matches (< 1e-10): 0 (0.0%)
Max absolute difference: 1.07e-01
Mean absolute difference: 5.54e-02

✅ Sample verification (first 10 tickers):
        Cache_ZScore  Calculated_ZScore  Difference
Ticker                                             
A             0.2354             0.1767     -0.0587
AA            0.4280             0.3663     -0.0617
AAL           1.0466             0.9751     -0.0715
AA

'zscore_verify_10d_20250102.csv'

In [ ]:
import pandas as pd
import numpy as np


def diagnose_price_gain_calculation(
    df_ohlcv, final_cache_df, verification_date, ticker="A", lookback=10
):
    """
    Diagnose the exact calculation method by testing multiple hypotheses.
    """

    verif_date = pd.Timestamp(verification_date)

    # Get dates
    all_dates = df_ohlcv.index.get_level_values("Date").unique().sort_values()
    end_idx = (
        all_dates.get_loc(verif_date)
        if verif_date in all_dates
        else all_dates.get_indexer([verif_date], method="nearest")[0]
    )
    start_idx = end_idx - lookback
    start_date = all_dates[start_idx]

    print(f"{'='*70}")
    print(f"DIAGNOSING {lookback}d_Price_Gain for {ticker} on {verif_date.date()}")
    print(f"Window: {start_date.date()} to {verif_date.date()}")
    print(f"{'='*70}")

    # Get prices
    idx = pd.IndexSlice
    ticker_prices = df_ohlcv.loc[idx[ticker, start_date:verif_date], "Adj Close"]

    print(f"\n📊 Raw Price Data ({len(ticker_prices)} points):")
    print(ticker_prices.to_string())

    p0 = ticker_prices.iloc[0]
    p_end = ticker_prices.iloc[-1]

    # Get cache Z-score
    cache_z = final_cache_df.loc[(verif_date, ticker), f"{lookback}d_Price_Gain"]
    print(f"\n🎯 Target Cache Z-Score: {cache_z:.6f}")

    # Test different return calculations
    methods = {}

    # Method 1: Simple return (P_end / P_start) - 1
    simple_return = (p_end / p0) - 1.0
    methods["Simple (P_end/P_start - 1)"] = simple_return

    # Method 2: Log return ln(P_end / P_start)
    log_return = np.log(p_end / p0)
    methods["Log ln(P_end/P_start)"] = log_return

    # Method 3: Cumulative sum of daily log returns
    daily_log_returns = np.log(ticker_prices / ticker_prices.shift(1)).dropna()
    cum_log_return = daily_log_returns.sum()
    methods["Cum Log Returns"] = cum_log_return

    # Method 4: Mean of daily simple returns * lookback
    daily_simple = ticker_prices.pct_change().dropna()
    mean_daily_simple = daily_simple.mean() * lookback
    methods["Mean Daily Simple * 10"] = mean_daily_simple

    # Method 5: Sum of daily simple returns
    sum_daily_simple = daily_simple.sum()
    methods["Sum Daily Simple"] = sum_daily_simple

    # Method 6: Annualized return scaled to 10d
    ann_return = ((p_end / p0) ** (252 / 10)) - 1
    methods["Annualized (10d scale)"] = ann_return * (10 / 252)  # De-annualize

    # Method 7: Excess return (minus risk-free or benchmark)
    # Would need benchmark data

    print(f"\n📊 Return Calculation Methods:")
    for name, value in methods.items():
        print(f"  {name:30s}: {value:+.6f}")

    # Now we need to reverse-engineer what cross-sectional stats would give cache_z
    # Z = (X - mean) / std  =>  mean = X - Z*std

    # Assuming std is similar to what we see in the market...
    assumed_stds = [0.03, 0.046, 0.05, 0.08, 0.10]

    print(f"\n📊 Reverse Engineering (what stats would produce Z={cache_z:.6f}):")
    for method_name, method_return in methods.items():
        implied_mean = method_return - cache_z * 0.046  # Using your observed std
        print(f"\n  {method_name}:")
        print(f"    Return: {method_return:+.6f}")
        print(f"    Implied cross-sectional mean: {implied_mean:+.6f}")

        # Test with different std assumptions
        for std in assumed_stds:
            z = (method_return - (-0.027)) / std  # Using -0.027 as example mean
            print(f"    If mean=-0.027, std={std:.3f}: Z={z:+.6f}")

    # Check if it's based on mean of prices or returns
    print(f"\n📊 Alternative: Based on Price Level?")
    price_mean = ticker_prices.mean()
    print(f"  Mean price over period: ${price_mean:.4f}")
    print(f"  (Not typically used for returns)")

    # Check _build_observation logic - uses lookback_returns
    print(f"\n📊 Checking _build_observation logic:")
    print(f"  lookback_close.iloc[0] (P0):  ${p0:.4f}")
    print(f"  lookback_close.iloc[-1] (P10): ${p_end:.4f}")
    print(f"  lookback_returns = pct_change():")
    print(daily_simple.to_string())
    print(f"  Mean of daily returns: {daily_simple.mean():.6f}")
    print(f"  Std of daily returns:  {daily_simple.std(ddof=1):.6f}")

    # The key: what does METRIC_REGISTRY['Price_Gain'] do?
    # Looking at your code, it likely uses the observation data

    return methods


def verify_all_tickers_detailed(
    df_ohlcv, final_cache_df, verification_date, lookback=10
):
    """
    Detailed verification trying to match the exact calculation.
    """

    verif_date = pd.Timestamp(verification_date)
    all_dates = df_ohlcv.index.get_level_values("Date").unique().sort_values()
    end_idx = (
        all_dates.get_loc(verif_date)
        if verif_date in all_dates
        else all_dates.get_indexer([verif_date], method="nearest")[0]
    )
    start_idx = end_idx - lookback
    start_date = all_dates[start_idx]

    idx = pd.IndexSlice

    # Get all tickers from cache for this date
    cache_slice = final_cache_df.loc[verif_date]
    all_tickers = cache_slice.index.tolist()

    print(f"Verifying {len(all_tickers)} tickers...")

    # Calculate raw returns using SIMPLE method (most likely)
    raw_returns = []
    for ticker in all_tickers:
        try:
            prices = df_ohlcv.loc[idx[ticker, start_date:verif_date], "Adj Close"]
            if len(prices) < 2:
                continue
            p0, p_end = prices.iloc[0], prices.iloc[-1]
            simple_ret = (p_end / p0) - 1.0
            log_ret = np.log(p_end / p0)
            raw_returns.append(
                {
                    "Ticker": ticker,
                    "Simple_Return": simple_ret,
                    "Log_Return": log_ret,
                    "Cache_Z": cache_slice.loc[ticker, f"{lookback}d_Price_Gain"],
                }
            )
        except:
            continue

    df = pd.DataFrame(raw_returns).set_index("Ticker")

    # Calculate Z-scores both ways
    df["Z_Simple"] = (df["Simple_Return"] - df["Simple_Return"].mean()) / df[
        "Simple_Return"
    ].std(ddof=1)
    df["Z_Log"] = (df["Log_Return"] - df["Log_Return"].mean()) / df["Log_Return"].std(
        ddof=1
    )

    # Compare
    df["Diff_Simple"] = (df["Z_Simple"] - df["Cache_Z"]).abs()
    df["Diff_Log"] = (df["Z_Log"] - df["Cache_Z"]).abs()

    print(f"\n{'='*70}")
    print(f"VERIFICATION SUMMARY")
    print(f"{'='*70}")
    print(f"Simple Return Z-Score vs Cache:")
    print(f"  Mean absolute difference: {df['Diff_Simple'].mean():.6f}")
    print(f"  Max absolute difference:  {df['Diff_Simple'].max():.6f}")
    print(f"  Exact matches (<1e-10):   {(df['Diff_Simple'] < 1e-10).sum()}/{len(df)}")

    print(f"\nLog Return Z-Score vs Cache:")
    print(f"  Mean absolute difference: {df['Diff_Log'].mean():.6f}")
    print(f"  Max absolute difference:  {df['Diff_Log'].max():.6f}")
    print(f"  Exact matches (<1e-10):   {(df['Diff_Log'] < 1e-10).sum()}/{len(df)}")

    # Show worst mismatches for simple
    if df["Diff_Simple"].max() > 1e-6:
        print(f"\nWorst Simple Return mismatches:")
        print(
            df.nlargest(5, "Diff_Simple")[
                ["Simple_Return", "Z_Simple", "Cache_Z", "Diff_Simple"]
            ]
        )

    return df


#

In [67]:
# First diagnose with single ticker
diagnose_price_gain_calculation(
    df_ohlcv, final_cache_df, "2025-01-02", ticker="A", lookback=10
)

# Then verify all
results = verify_all_tickers_detailed(
    df_ohlcv, final_cache_df, "2025-01-02", lookback=10
)

DIAGNOSING 10d_Price_Gain for A on 2025-01-02
Window: 2024-12-17 to 2025-01-02

📊 Raw Price Data (11 points):
Ticker  Date      
A       2024-12-17    135.121
        2024-12-18    131.704
        2024-12-19    131.883
        2024-12-20    133.190
        2024-12-23    133.279
        2024-12-24    134.764
        2024-12-26    134.497
        2024-12-27    134.210
        2024-12-30    133.101
        2024-12-31    133.267
        2025-01-02    132.364

🎯 Target Cache Z-Score: 0.235365

📊 Return Calculation Methods:
  Simple (P_end/P_start - 1)    : -0.020404
  Log ln(P_end/P_start)         : -0.020615
  Cum Log Returns               : -0.020615
  Mean Daily Simple * 10        : -0.020116
  Sum Daily Simple              : -0.020116
  Annualized (10d scale)        : -0.016079

📊 Reverse Engineering (what stats would produce Z=0.235365):

  Simple (P_end/P_start - 1):
    Return: -0.020404
    Implied cross-sectional mean: -0.031231
    If mean=-0.027, std=0.030: Z=+0.219869
    If mea

In [57]:
import pandas as pd
import numpy as np
from datetime import timedelta


def verify_zscore_calculation(df_ohlcv, final_cache_df, verification_date, lookback=10):
    """
    Verify Z-score calculation for all tickers by:
    1. Calculating raw 10d returns from df_ohlcv
    2. Computing cross-sectional mean and std
    3. Reproducing Z-scores
    4. Comparing with final_cache_df
    """

    verif_date = pd.Timestamp(verification_date)

    # Get all dates from ohlcv
    all_dates = df_ohlcv.index.get_level_values("Date").unique().sort_values()

    # Find exact or nearest date
    if verif_date not in all_dates:
        nearest_idx = all_dates.get_indexer([verif_date], method="nearest")[0]
        actual_date = all_dates[nearest_idx]
        print(f"⚠️  Using nearest date: {actual_date.date()}")
    else:
        actual_date = verif_date

    # Calculate lookback window (need lookback+1 days for proper return calc)
    date_list = all_dates.tolist()
    end_idx = (
        date_list.index(actual_date)
        if actual_date in date_list
        else [
            i
            for i, d in enumerate(date_list)
            if str(d.date()) == str(actual_date.date())
        ][0]
    )

    start_idx = end_idx - lookback

    if start_idx < 0:
        raise ValueError(f"Not enough history for {lookback}d calculation")

    start_date = date_list[start_idx]

    print(f"\n{'='*70}")
    print(f"Z-SCORE VERIFICATION: {lookback}d_Price_Gain for {actual_date.date()}")
    print(f"Lookback: {start_date.date()} to {actual_date.date()}")
    print(f"{'='*70}")

    # Step 1: Calculate raw returns for ALL tickers from df_ohlcv
    print(f"\n📊 Step 1: Calculating raw {lookback}d returns from df_ohlcv...")

    idx = pd.IndexSlice

    # Get all tickers available on both dates
    tickers_start = (
        df_ohlcv.loc[idx[:, start_date], :].index.get_level_values("Ticker").unique()
    )
    tickers_end = (
        df_ohlcv.loc[idx[:, actual_date], :].index.get_level_values("Ticker").unique()
    )
    all_tickers = tickers_start.intersection(tickers_end)

    print(f"   Tickers with data on both dates: {len(all_tickers)}")

    raw_returns = []

    for ticker in all_tickers:
        try:
            # Get prices
            p0 = df_ohlcv.loc[idx[ticker, start_date], "Adj Close"]
            p_end = df_ohlcv.loc[idx[ticker, actual_date], "Adj Close"]

            # Handle both scalar and Series (in case of duplicates)
            if isinstance(p0, pd.Series):
                p0 = p0.iloc[0]
            if isinstance(p_end, pd.Series):
                p_end = p_end.iloc[0]

            raw_return = (p_end / p0) - 1.0
            raw_returns.append(
                {"Ticker": ticker, f"{lookback}d_Raw_Return": raw_return}
            )

        except (KeyError, IndexError):
            continue

    raw_df = pd.DataFrame(raw_returns).set_index("Ticker")
    print(f"   Successfully calculated: {len(raw_df)} raw returns")
    print(
        f"   Raw return stats: mean={raw_df[f'{lookback}d_Raw_Return'].mean():.4f}, "
        f"std={raw_df[f'{lookback}d_Raw_Return'].std(ddof=1):.4f}"
    )

    # Step 2: Get Z-scores from final_cache_df
    print(f"\n📊 Step 2: Getting Z-scores from final_cache_df...")

    try:
        cache_slice = final_cache_df.loc[actual_date]
        z_col = f"{lookback}d_Price_Gain"

        if z_col not in cache_slice.columns:
            print(f"❌ Column {z_col} not found in final_cache_df")
            return None

        cache_zscores = cache_slice[[z_col]].copy()
        cache_zscores.columns = ["Cache_ZScore"]
        print(f"   Found {len(cache_zscores)} Z-scores in cache")

    except KeyError:
        print(f"❌ Date {actual_date} not found in final_cache_df")
        return None

    # Step 3: Reproduce Z-scores from raw returns
    print(f"\n📊 Step 3: Reproducing Z-scores...")

    # Calculate cross-sectional statistics (must match the worker_task logic)
    mean_return = raw_df[f"{lookback}d_Raw_Return"].mean()
    std_return = raw_df[f"{lookback}d_Raw_Return"].std(ddof=1)  # ddof=1 for sample std

    print(f"   Cross-sectional mean: {mean_return:.6f}")
    print(f"   Cross-sectional std (ddof=1): {std_return:.6f}")

    # Calculate Z-scores: (X - mean) / std
    raw_df["Calculated_ZScore"] = (
        raw_df[f"{lookback}d_Raw_Return"] - mean_return
    ) / std_return

    # Step 4: Merge and compare
    print(f"\n📊 Step 4: Comparing calculations...")

    comparison = raw_df.join(cache_zscores, how="inner")
    comparison["Difference"] = (
        comparison["Calculated_ZScore"] - comparison["Cache_ZScore"]
    )
    comparison["Abs_Difference"] = comparison["Difference"].abs()

    # Statistics
    max_diff = comparison["Abs_Difference"].max()
    mean_diff = comparison["Abs_Difference"].mean()
    exact_matches = (comparison["Abs_Difference"] < 1e-10).sum()

    print(f"\n{'='*70}")
    print(f"VERIFICATION RESULTS")
    print(f"{'='*70}")
    print(f"Tickers compared: {len(comparison)}")
    print(
        f"Exact matches (< 1e-10): {exact_matches} ({100*exact_matches/len(comparison):.1f}%)"
    )
    print(f"Max absolute difference: {max_diff:.2e}")
    print(f"Mean absolute difference: {mean_diff:.2e}")

    # Show worst mismatches
    if max_diff > 1e-6:
        print(f"\n⚠️  Largest discrepancies (top 5):")
        worst = comparison.nlargest(5, "Abs_Difference")
        print(worst[["Cache_ZScore", "Calculated_ZScore", "Difference"]].to_string())

    # Show sample of good matches
    print(f"\n✅ Sample verification (first 10 tickers):")
    sample = comparison.head(10)[["Cache_ZScore", "Calculated_ZScore", "Difference"]]
    print(sample.to_string())

    # Step 5: Detailed Excel-style verification for specific tickers
    print(f"\n{'='*70}")
    print(f"DETAILED BREAKDOWN (First 3 Tickers)")
    print(f"{'='*70}")

    for ticker in comparison.head(3).index:
        row = comparison.loc[ticker]
        print(f"\n{ticker}:")
        print(f"  Raw Return:     {row[f'{lookback}d_Raw_Return']:.6f}")
        print(f"  Mean (all):     {mean_return:.6f}")
        print(f"  Std (all):      {std_return:.6f}")
        print(
            f"  Calculation:    ({row[f'{lookback}d_Raw_Return']:.6f} - {mean_return:.6f}) / {std_return:.6f}"
        )
        print(
            f"                  = {row[f'{lookback}d_Raw_Return'] - mean_return:.6f} / {std_return:.6f}"
        )
        print(f"                  = {row['Calculated_ZScore']:.6f}")
        print(f"  Cache Z-Score:  {row['Cache_ZScore']:.6f}")
        print(f"  Match:          {'✅' if row['Abs_Difference'] < 1e-10 else '❌'}")

    return comparison


def export_verification_csv(
    comparison_df, verification_date, lookback=10, output_file=None
):
    """
    Export detailed verification to CSV for Excel checking.
    """
    if output_file is None:
        output_file = f"zscore_verify_{lookback}d_{pd.Timestamp(verification_date).strftime('%Y%m%d')}.csv"

    # Add verification formula columns
    export_df = comparison_df.copy()

    # Excel formula hint
    z_col = f"{lookback}d_Raw_Return"
    mean_return = export_df[z_col].mean()
    std_return = export_df[z_col].std(ddof=1)

    export_df["Excel_Verification_Mean"] = mean_return
    export_df["Excel_Verification_Std"] = std_return
    export_df["Excel_Formula"] = f"=(B2-${mean_return:.6f})/${std_return:.6f}"

    # Reorder columns nicely
    cols = [
        z_col,
        "Excel_Verification_Mean",
        "Excel_Verification_Std",
        "Calculated_ZScore",
        "Cache_ZScore",
        "Difference",
    ]

    export_df[cols].to_csv(output_file)
    print(f"\n💾 Detailed verification saved: {output_file}")

    return output_file


#

In [58]:
# Verify all 926 tickers for 10d_Price_Gain on 2025-01-02
verification_results = verify_zscore_calculation(
    df_ohlcv=df_ohlcv,
    final_cache_df=final_cache_df,
    verification_date="2025-01-02",
    lookback=10,
)

# Export to CSV for Excel verification
export_verification_csv(verification_results, "2025-01-02", 10)


Z-SCORE VERIFICATION: 10d_Price_Gain for 2025-01-02
Lookback: 2024-12-17 to 2025-01-02

📊 Step 1: Calculating raw 10d returns from df_ohlcv...
   Tickers with data on both dates: 1555
   Successfully calculated: 1555 raw returns
   Raw return stats: mean=-0.0274, std=0.0464

📊 Step 2: Getting Z-scores from final_cache_df...
   Found 926 Z-scores in cache

📊 Step 3: Reproducing Z-scores...
   Cross-sectional mean: -0.027377
   Cross-sectional std (ddof=1): 0.046448

📊 Step 4: Comparing calculations...

VERIFICATION RESULTS
Tickers compared: 926
Exact matches (< 1e-10): 0 (0.0%)
Max absolute difference: 2.34e+00
Mean absolute difference: 7.92e-02

⚠️  Largest discrepancies (top 5):
        Cache_ZScore  Calculated_ZScore  Difference
Ticker                                             
TSLL        -10.2253            -7.8894      2.3359
IREN         -5.2440            -4.5382      0.7058
MSTR         -4.8317            -4.2249      0.6068
CVNA         -4.5468            -4.0050      0.541

'zscore_verify_10d_20250102.csv'

In [ ]:
#############################
#############################

#### Build final_cache_df 

In [ ]:
# 2. THE STITCH (Run this when the progress bar hits 100%)
final_cache_df = FeatureCubeStitcher.assemble(CHECKPOINT_DIR, FINAL_FILE)

In [ ]:
list(final_cache_df.columns)

In [ ]:
final_cache_df

In [ ]:
final_cache_df.info()

In [ ]:
final_cache_df.describe()

#### Save final_cache_df 

In [ ]:
# Get first and last dates from the index (level 0 is the date index)
first_date = final_cache_df.index.get_level_values(0).min().strftime("%Y%m%d")
last_date = final_cache_df.index.get_level_values(0).max().strftime("%Y%m%d")

# Construct filename with date range
fn_final_cache_df = OUTPUT_DIR / f"final_cache_df_{first_date}_{last_date}.parquet"
print(f"final_cache_df file name: {fn_final_cache_df}")

final_cache_df.to_parquet(fn_final_cache_df, engine="pyarrow", compression="zstd")
print(f"Saved final_cache_df to:\n{fn_final_cache_df}")

#### Verify saved final_cache_df 

In [ ]:
# Load fn_final_cache_df to a different variable name
final_cache_df_reloaded = pd.read_parquet(fn_final_cache_df, engine="pyarrow")
print(f"\nLoaded: {fn_final_cache_df}\nTo: final_cache_df_reloaded\n")
print(f"final_cache_df_reloaded.head():\n{final_cache_df_reloaded.head()}")
print(f"final_cache_df_reloaded.tail():\n{final_cache_df_reloaded.tail()}")

# Verify data integrity
print(
    f"\nShape check - Original: {final_cache_df.shape}, Reloaded: {final_cache_df_reloaded.shape}"
)
print(f"Index match: {final_cache_df.index.equals(final_cache_df_reloaded.index)}")
print(
    f"Columns match: {final_cache_df.columns.equals(final_cache_df_reloaded.columns)}"
)

# Release from memory
del final_cache_df_reloaded

gc.collect()
print("\nReleased final_cache_df_reloaded from memory")

In [ ]:
#############################
#############################

#### Load final_cache_df

In [ ]:
fn_batch_df = r"C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\_alpha_cache_v1_checkpoints\batch_20260226.parquet"
# Load fn_batch_df to a different variable name
batch_df = pd.read_parquet(fn_batch_df, engine="pyarrow")
print(f"\nLoaded: {fn_batch_df}\nTo: batch_df\n")
print(f"batch_df.head():\n{batch_df.head()}")
print(f"batch_df.tail():\n{batch_df.tail()}")

In [ ]:
list(batch_df.columns)

In [ ]:
fn_final_cache_df = OUTPUT_DIR / "final_cache_df_20060901_20260320.parquet"
# Load fn_final_cache_df to a different variable name
final_cache_df = pd.read_parquet(fn_final_cache_df, engine="pyarrow")
print(f"\nLoaded: {fn_final_cache_df}\nTo: final_cache_df\n")
print(f"final_cache_df.head():\n{final_cache_df.head()}")
print(f"final_cache_df.tail():\n{final_cache_df.tail()}")

In [ ]:
# Or if you want to modify in-place (more memory efficient)
final_cache_df.rename(columns=lambda x: x.replace(",", ""), inplace=True)
final_cache_df.info()

In [ ]:
######################
######################

In [ ]:
# 6. Instantiate engine (customize DataFrames as needed)
old_master_engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)

In [ ]:
# 1. Define the Action (Inputs)
my_input = EngineInput(
    mode="Ranking",
    start_date=pd.Timestamp("2025-12-10"),  # Decision Date from your screenshot
    lookback_period=21,
    holding_period=5,
    metric="Sharpe (ATRP)",
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
    rank_start=1,
    rank_end=10,
    debug=True,
)

# 2. Run the Headless Engine
metrics_df = run_headless_simulation(old_master_engine, my_input)

# 3. Verify Result
print("--- HEADLESS METRICS REPORT ---")
display(metrics_df)

# TEST: Gain in Holding Period (The Reward for RL)
reward = metrics_df.loc["Group Gain", "Holding"]
print(f"\nTarget Reward Signal: {reward:.4f}")

In [ ]:
######################
######################

In [ ]:
class RLVRParityVerifier:
    @staticmethod
    def verify(
        cache_df: pd.DataFrame,
        df_close_wide: pd.DataFrame,
        engine_output: any,
        lookback: int,
        registry_metric: str,
        top_n: int = 10,
    ):
        # 1. Modular Name Mapping
        # Slugify returns a list, so we take the first element
        clean_metric = AlphaLogic.slugify_columns([registry_metric])[0]
        print(f"\nregistry_metric: {registry_metric}")
        print(f"clean_metric: {clean_metric}")

        cache_col = f"{lookback}d_{clean_metric}"
        decision_date = engine_output.decision_date
        print(f"cache_col: {cache_col}")
        print(f"decision_date: {decision_date}")

        # 2. Selection Parity
        day_slice = cache_df.xs(decision_date, level="Date")[cache_col]
        cache_tickers = (
            day_slice.sort_values(ascending=False).head(top_n).index.tolist()
        )
        print(f"day_slice cache_df: {day_slice}")
        print(f"cache_tickers: {cache_tickers}")

        engine_tickers = engine_output.tickers
        print(f"engine_tickers: {engine_tickers}")

        # 3. REWARD LOGIC: REPLICATING QuantUtils.compute_portfolio_stats
        # Entry (T+1) to End (T+1+H)
        entry_date = engine_output.buy_date
        end_date = engine_output.holding_end_date

        # Slice the window [Entry : End]
        price_window = df_close_wide.loc[entry_date:end_date, cache_tickers]
        print(f"price_window cache_tickers:\n{price_window}\n")

        # a) Normalize prices to the START of the holding period (Entry Date)
        # This is what norm_prices = prices.div(prices.bfill().iloc[0]) does
        norm_prices = price_window.div(price_window.iloc[0])

        # b) Equal weights at the start (1/N)
        weights = 1.0 / len(cache_tickers)

        # c) Equity Curve = Sum of (Relative Price * Weight)
        # This accounts for the "Price Drift" within the holding period
        equity_curve = (norm_prices * weights).sum(axis=1)

        # d) Calculate Total Period Log Return (Veritable Reward)
        final_equity = equity_curve.iloc[-1]
        vectorized_log_reward = np.log(final_equity)

        # 4. Comparison
        legacy_reward = engine_output.perf_metrics.get("holding_p_gain", 0.0)
        gain_diff = abs(vectorized_log_reward - legacy_reward)

        print(f"--- 100% PARITY REPORT: {decision_date.date()} ---")
        print(
            f"Ticker Match:         {'✅ PASS' if set(cache_tickers) == set(engine_tickers) else '❌ FAIL'}"
        )
        print(f"Log Reward Match:     {'✅ PASS' if gain_diff < 1e-9 else '❌ FAIL'}")
        print(f"  - Vectorized (Log): {vectorized_log_reward:.10f}")
        print(f"  - Legacy (Log):     {legacy_reward:.10f}")
        print(f"  - Delta:            {gain_diff:.2e}")


# # Run the test again
# RLVRParityVerifier.verify(
#     cache_df=final_cache_df,
#     df_close_wide=df_close_wide,
#     engine_output=engine_output,
#     lookback=21,
#     registry_metric="Sharpe (ATRP)",
#     top_n=10,
# )

In [ ]:
# for key in [
#     "Sharpe (TRP)",
#     "Momentum (21d)",
#     "Consistency (WinRate 5d)",
#     "Low Volatility (-ATRP)",
#     "VIX Filtered Momentum",
# ]:
for key in METRIC_REGISTRY.keys():
    # 1. Update the input metric to TRP
    my_input.metric = key
    my_input.rank_start = 1
    my_input.rank_end = 1
    print(f"my_input.metric: {my_input.metric}")

    # 2. RE-RUN the engine to get a FRESH output for TRP
    engine_output = old_master_engine.run(my_input)

    # 3. NOW run the parity check with the matching output
    RLVRParityVerifier.verify(
        cache_df=final_cache_df,
        df_close_wide=df_close_wide,
        engine_output=engine_output,  # Use the new output
        lookback=21,
        registry_metric=my_input.metric,
        top_n=1,
    )
    print(f'{"="*10}\n')

In [ ]:
# for key in [
#     "Sharpe (TRP)",
#     "Momentum (21d)",
#     "Consistency (WinRate 5d)",
#     "Low Volatility (-ATRP)",
#     "VIX Filtered Momentum",
# ]:
for key in METRIC_REGISTRY.keys():
    # 1. Update the input metric to TRP
    my_input.metric = key
    my_input.rank_start = 1
    my_input.rank_end = 1
    print(f"my_input.metric: {my_input.metric}")

    # 2. RE-RUN the engine to get a FRESH output for TRP
    engine_output = old_master_engine.run(my_input)

    # 3. NOW run the parity check with the matching output
    RLVRParityVerifier.verify(
        cache_df=final_cache_df,
        df_close_wide=df_close_wide,
        engine_output=engine_output,  # Use the new output
        lookback=63,
        registry_metric=my_input.metric,
        top_n=1,
    )
    print(f'{"="*10}\n')

In [ ]:
# Run the test again
RLVRParityVerifier.verify(
    cache_df=final_cache_df,
    df_close_wide=df_close_wide,
    engine_output=engine_output,
    lookback=21,
    registry_metric="Sharpe (ATRP)",
    top_n=10,
)

In [ ]:
final_cache_df.columns

In [ ]:
# 1. Update the input metric to TRP
my_input.metric = "Sharpe (TRP)"
print(f"my_input.metric: {my_input.metric}\n")

# 2. RE-RUN the engine to get a FRESH output for TRP
engine_output_trp = old_master_engine.run(my_input)

# 3. NOW run the parity check with the matching output
RLVRParityVerifier.verify(
    cache_df=final_cache_df,
    df_close_wide=df_close_wide,
    engine_output=engine_output_trp,  # Use the new output
    lookback=21,
    registry_metric=my_input.metric,
    top_n=10,
)

In [ ]:
######################
######################

In [ ]:
# List of metric names
metric_names = list(METRIC_REGISTRY.keys())
print(f"metric_names: {metric_names}")

In [ ]:
decision_date = pd.Timestamp("2025-12-10")
# decision_date = pd.Timestamp("2025-12-07")
# start_date = pd.Timestamp("2025-11-25")
holding_period = 5
lookback_period = 21
metric = "Sharpe (ATRP)"
rank_start = 11
rank_end = 12
benchmark_ticker = GLOBAL_SETTINGS["benchmark_ticker"]
thresholds = GLOBAL_SETTINGS["thresholds"]

calendar = new_master_engine.trading_calendar

# Use searchsorted to find the insertion point (next date if exact not found)
decision_idx = calendar.searchsorted(decision_date)

# Safety check: ensure we haven't requested a date beyond the calendar
if decision_idx >= len(calendar):
    raise ValueError(
        f"Decision date {decision_date.date()} is beyond the last trading date "
        f"({calendar[-1].date()})"
    )

actual_trading_date = calendar[decision_idx]

# Only print and replace if the requested date is not in the index
if actual_trading_date != decision_date:
    print(
        f"=== Using decision_date: {actual_trading_date.date()} ===\n"
        f"=== Requested decision_date: {decision_date.date()} is not in index ==="
    )
    decision_date = actual_trading_date

# Compare with Engine
engine_input = EngineInput(
    mode="Cascade",
    benchmark_ticker=benchmark_ticker,
    rank_start=rank_start,
    rank_end=rank_end,
    # start_date is decision_date
    start_date=decision_date,
    holding_period=holding_period,
    lookback_period=lookback_period,
    metric=metric,
    debug=True,
)
engine_out = new_master_engine.run(engine_input)

# # AlphaCache_Final run parameters
# lookback_periods = [21, 63, 189]
# lookback_period = lookback_periods[0]
# holding_period = 5
# metric = "Sharpe (ATRP)"

In [ ]:
engine_out.perf_metrics
engine_out.tickers
# engine_out

In [ ]:
# 1. First, get the 'Survivor Pool' (The candidates) for this specific date
# This uses your verified liquidity/quality thresholds
candidates = new_master_engine._filter_universe(
    date_ts=decision_date,
    thresholds=thresholds,
    audit_container={},  # We pass an empty dict for headless runs
)
print(f"len(candidates): {len(candidates)}")

In [ ]:
# # decision_date = pd.Timestamp("2025-01-11")
# decision_idx = new_master_engine.trading_calendar.get_loc(decision_date)
# decision_idx

In [ ]:
# --- THE PRECISION PATTERN ---

# 1. Find where 'Today' (Decision Date) sits in the calendar
decision_idx = new_master_engine.trading_calendar.get_loc(decision_date)

# 2. Go back exactly 21 steps to find the Anchor (P0)
# This handles holidays (Christmas, New Year) automatically
start_idx = decision_idx - lookback_period
start_date = new_master_engine.trading_calendar[start_idx]

# print(f"Decision Date: {pd.Timestamp('2025-01-13').date()}")
print(f"Decision Date: {decision_date.date()}")
print(f"Precise P0 Anchor (21 days back): {start_date.date()}")

In [ ]:
print(f"decision_date: {decision_date.date()}")
print(f"start_date: {start_date.date()}")
print(f"len(candidates): {len(candidates)}")

In [ ]:
# 1. Initialize our storage for this date's ensemble
ensemble_parts = []

# 2. Outer Loop: Resolutions (Lookbacks)
# for lb in [21, 63, 189]:
for lb in [lookback_period]:
    # Generate the 'MarketObservation' for this specific resolution
    # This is where 'obs' is born
    # obs = new_master_engine._build_observation(date, candidates, start_date_for_lb)
    obs = new_master_engine._build_observation(
        decision_date=decision_date, candidates=candidates, start_date=start_date
    )

    # 3. Inner Loop: The Metric Registry (The "Brain" Loop)
    for metric_name, metric_func in METRIC_REGISTRY.items():

        # --- THIS IS THE CALL ---
        # metric_func is the reference from the dictionary
        # obs is the dataclass containing prices, returns, ATRP, etc.
        score_series = metric_func(obs)

        # 4. Standardize the Column Name for the Agent
        # Example: '21d_Sharpe_ATRP'
        col_name = f"{lb}d_{metric_name}"
        score_series.name = col_name

        ensemble_parts.append(score_series)

# 5. Combine into a Single Matrix [Tickers x 33 Metrics]
ensemble_df = pd.concat(ensemble_parts, axis=1)

# 6. Apply to ensemble_df columns (static method called from outside the class)
ensemble_df.columns = AlphaLogic.slugify_columns(ensemble_df.columns.tolist())

In [ ]:
print(f"ensemble_df:\n{ensemble_df}")

In [ ]:
# ensemble_df
type(obs)
obs.lookback_returns.A

In [ ]:
# --- THE FORENSIC PRINT PATTERN ---

for metric_name, metric_func in METRIC_REGISTRY.items():
    # Execute the vectorized strategy
    scores = metric_func(obs)

    # 1. Look at the aggregate (The 'Market Weather')
    avg_val = scores.mean()
    count = len(scores)

    # 2. Look at the leaders (The 'Top Alpha')
    top_3 = scores.sort_values(ascending=False).head(3)

    print(f"📊 Strategy: {metric_name} ({count} tickers)")
    print(f"   Market Average: {avg_val:+.4f}")
    print(
        f"   Top 3 Tickers:  {top_3.index.tolist()} -> {top_3.values} -> {top_3.values.mean()}"
    )
    print("-" * 30)

In [ ]:
# Output file name for AlphaCache
fn_AlphaCache = OUTPUT_DIR / "AlphaCache_Final.parquet"
print(f"AlphaCache file name: {fn_AlphaCache}")

In [ ]:
# Load AlphaCache_Final.parquet
fn_AlphaCache = OUTPUT_DIR / "AlphaCache_Final.parquet"
final_cache_df = pd.read_parquet(fn_AlphaCache, engine="pyarrow")
print(f"\nLoaded: {fn_AlphaCache}\nTo: final_cache_df\n")
print(f"final_cache_df.head():\n{final_cache_df.head()}")
print(f"final_cache_df.tail():\n{final_cache_df.tail()}")

In [ ]:
print(f"final_cache_df columns:")
for index, value in enumerate(final_cache_df.columns):
    print(f"{index}: {value}")

In [ ]:
print(f"lookback_periods: {lookback_periods}")
print(f"start_date: {start_date.date()}")

In [ ]:
# --- THE MARATHON CONFIG ---
CHECKPOINT_DIR = "alpha_cache_v1_checkpoints"
FINAL_FILE = "AlphaCache_Master_2000_2026.parquet"

# 1. THE BAKE (Parallel + Resilient)
ParallelFeatureBuilder.run_marathon(
    new_master_engine=new_master_engine,
    lookbacks=[21, 63, 189],
    start_date="2000-01-01",
    checkpoint_dir=CHECKPOINT_DIR,
    batch_size=10,  # Saves to disk every 50 days
)

# 2. THE STITCH (Run this when the progress bar hits 100%)
# final_cache_df = FeatureCubeStitcher.assemble(CHECKPOINT_DIR, FINAL_FILE)

In [ ]:
# Create AlphaCache_Final takes 1.5 hour run
# 1. BAKE THE FINAL CUBE (The 1.5 Hour Re-run)
final_cache_df = FeatureCubeBuilder.build(
    new_master_engine=new_master_engine,
    lookbacks=lookback_periods,
    start_date="2000-01-01",
)

final_cache_df.to_parquet(fn_AlphaCache, engine="pyarrow", compression="zstd")
print(f"Saved final_cache_df to:\n{fn_AlphaCache}")

In [ ]:
new_master_engine.compute_alpha_ensemble(
    decision_date=start_date,
    lookback_periods=lookback_periods,
)

In [ ]:
# 1. Choose your Discovery resolution (Standard: 5 days)
# holding_pd = 5

# 2. Command the Engine to calculate the 'Forward-Truth'
# This generates the reward_matrix [Dates x Tickers]
# Logic: ln(Price_t+6 / Price_t+1)
new_master_engine.precompute_reward_matrix(holding_period=holding_period)

print(f"✅ Reward Matrix Generated. Shape: {new_master_engine.reward_matrix.shape}")

# 3. NOW Initialize the Environment
# It will now find 'new_master_engine.reward_matrix' and work perfectly.
env = DiscoveryEnv(
    feature_cube=final_cache_df,
    reward_matrix=new_master_engine.reward_matrix,
    calendar=new_master_engine.trading_calendar,
    holding_period=holding_period,
)

print("🚀 DiscoveryEnv Ready. Proceed to One-Hot Audit.")

In [ ]:
engine_out.perf_metrics

In [ ]:
# # 1. BAKE THE FINAL CUBE (The 1.5 Hour Re-run)
# # Use start_date="2024-12-10" as we discussed
# final_cache_df = FeatureCubeBuilder.build(
#     new_master_engine, [21, 63, 189], pd.Timestamp("2024-12-10")
# )
# # final_cache_df.to_pickle("AlphaCache_Final_Slugified.pkl")
# final_cache_df.to_parquet(f_name, engine="pyarrow", compression="zstd")
# print(f"Saved final_cache_df to:\n{f_name}")

# 2. INITIALIZE THE NEW ENVIRONMENT
env = DiscoveryEnv(
    final_cache_df, new_master_engine.reward_matrix, new_master_engine.trading_calendar
)

# 3. RUN ONE-HOT AUDIT
# test_date = pd.Timestamp("2025-01-02")
target_metric = "21d_Sharpe_ATRP"  # Note: Slugified name!
metric_idx = final_cache_df.columns.get_loc(target_metric)

# Build One-Hot Action
action = np.zeros(final_cache_df.shape[1] + 2)
action[metric_idx] = 1.0
action[-2], action[-1] = -1.0, 1.0  # Offset 0, Width 10

env.reset(start_date=start_date)
_, env_reward, _, info = env.step(action)

# Compare with Engine
engine_input = EngineInput(
    mode="Cascade",
    benchmark_ticker="SPY",
    start_date=start_date,
    lookback_period=21,
    holding_period=5,
    metric="Sharpe (ATRP)",
    debug=True,
)
engine_out = new_master_engine.run(engine_input)

# SAVE EXCEL AUDIT
audit_df = engine_out.debug_data["selection_audit"].join(
    final_cache_df.xs(start_date, level="Date")[[target_metric]], how="inner"
)

fn_audit_df_csv = OUTPUT_DIR / "Final_Absolute_Zero_Audit.csv"
audit_df.to_csv(fn_audit_df_csv)
print(f"Save audit_df to: {fn_audit_df_csv}")

engine_reward = engine_out.perf_metrics["holding_p_gain"]
reward_diff = engine_reward - env_reward
print(f"✅ Audit Generated. Open 'Final_Absolute_Zero_Audit.csv'")
print(f"Env Reward: {env_reward:.10f}")
print(f"Engine Reward: {engine_reward:.10f}")
print(f"Reward Diff: {reward_diff:.10f}")

In [ ]:
# 1. Define the Discovery Goal (5-Day Holding)
holding_pd = 5

# 2. Precompute the "Truth" (The Reward Matrix)
# This calculates: (Price at Decision+1+5 / Price at Decision+1) - 1
new_master_engine.precompute_reward_matrix(holding_period=holding_pd)

print(f"✅ Reward Matrix Generated. Shape: {new_master_engine.reward_matrix.shape}")

# 3. NOW Initialize the Environment
# We pass the newly baked matrix into the stateful Arena
env = DiscoveryEnv(
    feature_cube=final_cache_df,
    reward_matrix=new_master_engine.reward_matrix,
    calendar=new_master_engine.trading_calendar,
    holding_period=holding_pd,
)

print("🚀 DiscoveryEnv Ready for Audit.")

In [ ]:
print(list(engine_out.perf_metrics.keys()))

In [ ]:
# --- THE FINAL PARITY AUDIT ---
test_date = pd.Timestamp("2025-01-02")
target_metric = "21d_Sharpe_ATRP"  # Verified Slugified Name

# 1. Build the One-Hot Action
# Index: 0 to 32 are metrics. 33 and 34 are Rank params.
metric_idx = final_cache_df.columns.get_loc(target_metric)
action = np.zeros(final_cache_df.shape[1] + 2)
action[metric_idx] = 1.0
action[-2], action[-1] = -1.0, 1.0  # Top 10 Tickers

# 2. Execute Step in the Arena
env.reset(start_date=test_date)
_, env_log_reward, _, info = env.step(action)

# 3. Execute the 'Old Guard' Human Engine
engine_input = EngineInput(
    mode="Cascade",
    benchmark_ticker="SPY",
    start_date=test_date,
    lookback_period=21,
    holding_period=holding_pd,
    metric="Sharpe (ATRP)",  # The original registry name
    debug=True,
)
engine_out = new_master_engine.run(engine_input)

# 4. Compare arithmetic vs log
eng_log_reward = engine_out.perf_metrics["holding_p_gain"]


print(f"\n--- AUDIT RESULT for {test_date.date()} ---")
print(f"Env Log Reward:     {env_log_reward:.12f}")
print(f"Engine Log Reward:  {eng_log_reward:.12f}")
print(f"Difference:         {abs(env_log_reward - eng_log_reward):.12f}")

# 5. Save the Excel Verification File
# This joins the Engine's Raw Scores with our Cache's Z-Scores
audit_df = engine_out.debug_data["selection_audit"].join(
    final_cache_df.xs(test_date, level="Date")[[target_metric]], how="inner"
)
audit_df.to_csv("Final_Absolute_Zero_Audit.csv")
print(f"💾 Final Audit Saved: 'Final_Absolute_Zero_Audit.csv'")

In [ ]:
from pathlib import Path

# 1. Define your pathing structure
# This ensures it works whether you are on a Local PC or Google Colab
cache_dir = Path("cache")
cache_file = cache_dir / "AlphaCache_2025_V1.pkl"

# 2. Safety First: Create the directory if it doesn't exist
# parents=True means it will create the whole path (e.g., data/discovery/cache)
# exist_ok=True means it won't crash if the folder is already there
cache_dir.mkdir(parents=True, exist_ok=True)

# 3. Save the 'Feature Cube'
# We use pickle because it preserves the Pandas MultiIndex and dtypes perfectly
try:
    cache.feature_cube.to_pickle(cache_file)
    print(f"✅ Persistence Successful!")
    print(f"💾 Path: {cache_file.absolute()}")
    print(f"📊 Cube Size: {cache.feature_cube.memory_usage().sum() / 1e6:.2f} MB")
except Exception as e:
    print(f"❌ Persistence Failed: {e}")

# 4. Quick Verify (The Trust Check)
# Ensure we can read it back immediately
import pandas as pd

test_load = pd.read_pickle(cache_file)
assert test_load.shape == cache.feature_cube.shape
print("🧪 Load-Back Test: PASSED. The data is safe.")

In [ ]:
import os
import pandas as pd
from core.environment import DiscoveryEnv

# 1. Define the Path
CACHE_PATH = os.path.join("cache", "AlphaCache_2025_V1.pkl")

# 2. Safety Check & Load
if not os.path.exists(CACHE_PATH):
    raise FileNotFoundError(f"❌ Could not find the Intelligence Cube at {CACHE_PATH}")

print(f"📂 Loading AlphaCache from {CACHE_PATH}...")
df_cube = pd.read_pickle(CACHE_PATH)

# 3. Re-inject into the Cache Object
# We pass the same lookbacks used during the 1.5-hour 'bake'
cache = AlphaCache(new_master_engine, lookbacks=[21, 63, 189])
cache.feature_cube = df_cube

print(f"✅ Intelligence Cube Restored: {cache.feature_cube.shape[0]} snapshots.")

# 4. Re-initialize the Discovery Environment
# This is our 'HFT-Speed' Training Gym
env = DiscoveryEnv(engine=new_master_engine, cache=cache, holding_period=5)

# 5. Heartbeat Verification
test_date = pd.Timestamp("2025-01-02")
obs = env.reset(start_date=test_date)
print(f"💓 System Heartbeat: Success. Current Date: {obs['date'].date()}")

In [ ]:
# 1. Setup Parameters
test_date = pd.Timestamp("2025-01-02")
target_metric_full = "21d_Sharpe (ATRP)"  # The Cache Name
target_metric_base = "Sharpe (ATRP)"  # The Engine Name
benchmark_ticker = "SPY"


print(f"🕵️ Starting Deep Audit for {test_date.date()}...")

# --- STEP A: THE HUMAN ENGINE (RAW TRUTH) ---
# We run the engine in 'debug' mode to get the full universe ranking
engine_input = EngineInput(
    benchmark_ticker="SPY",
    mode="Cascade",
    start_date=test_date,
    lookback_period=21,
    holding_period=5,
    metric=target_metric_base,
    rank_start=1,
    rank_end=10,
    debug=True,
)
engine_out = new_master_engine.run(engine_input)

# Get the full universe scores before slicing (from debug data)
raw_scores_df = engine_out.debug_data["selection_audit"]
# Let's keep Ticker, Strategy_Score (Raw), and Lookback_ATRP
engine_audit = raw_scores_df[["Strategy_Score", "Lookback_ATRP"]].rename(
    columns={"Strategy_Score": "Engine_Raw_Value"}
)

# --- STEP B: THE AI CACHE (Z-SCORED VISION) ---
# Grab the specific date slice from the cache
cache_slice = cache.get_vision(test_date)
# Get the specific Z-scored column
cache_audit = cache_slice[[target_metric_full]].rename(
    columns={target_metric_full: "Cache_Z_Score"}
)

# --- STEP C: THE MERGE (THE EXCEL MOMENT) ---
# Combine both views on the Ticker index
verification_df = engine_audit.join(cache_audit, how="inner")

# Add a 'Rank' column based on the Raw Value to verify sorting
verification_df["Engine_Rank"] = verification_df["Engine_Raw_Value"].rank(
    ascending=False
)

# Add a 'Rank' column based on the Z-Score (They should be IDENTICAL)
verification_df["Cache_Rank"] = verification_df["Cache_Z_Score"].rank(ascending=False)

# --- STEP D: THE REWARD VERIFICATION ---
# Get the reward calculated by the environment
one_hot_action = np.zeros(cache.feature_cube.shape[1] + 2)
one_hot_action[cache.feature_cube.columns.get_loc(target_metric_full)] = 1.0
one_hot_action[-2], one_hot_action[-1] = -1.0, 1.0  # Offset 0, Width 10

env.reset(start_date=test_date)
_, env_reward, _, env_info = env.step(one_hot_action)

# # --- FINAL OUTPUT ---
# print(f"✅ Audit Complete. Verification File Generated.")
# print(f"Engine Reward: {engine_out.perf_metrics['p_hold_ret_ann']:.6f} (Annualized)")
# print(f"Env Log Reward: {env_reward:.6f} (5-day Log)")

# --- FINAL OUTPUT (FIXED) ---
print("\n--- PERFORMANCE AUDIT ---")
# 1. Inspect what the Engine actually calculated
print(f"Available Metric Keys: {list(engine_out.perf_metrics.keys())}")

# 2. Get the Arithmetic Return from the Engine
# Based on our AlphaEngine logic, the key is likely 'p_hold_ret'
engine_arith_ret = engine_out.perf_metrics.get("holding_p_gain", 0.0)

# 3. Calculate what the Log Reward SHOULD be based on that Engine return
expected_log_reward = np.log1p(engine_arith_ret)

print(f"Engine Raw Period Return (Arithmetic): {engine_arith_ret:.6f}")
print(f"Expected Log Reward [LN(1+R)]:      {expected_log_reward:.6f}")
print(f"Environment Log Reward (Actual):     {env_reward:.6f}")

# 4. Precision Check
diff = abs(expected_log_reward - env_reward)
if diff < 1e-10:
    print(
        "✅ REWARD PARITY: The environment reward is a perfect Log-Mirror of the engine."
    )
else:
    print(f"⚠️ REWARD DISCREPANCY: Difference of {diff:.12f}")


# Save to CSV
verification_df.to_csv("AbsoluteZero_Verification_Audit.csv")
print(f"💾 File saved as 'AbsoluteZero_Verification_Audit.csv'. Open this in Excel!")

In [ ]:
list(cache_slice.columns)

In [ ]:
# # Let's keep Ticker, Strategy_Score (Raw), and Lookback_ATRP
# engine_audit = raw_scores_df[["Strategy_Score", "Lookback_ATRP"]].rename(
#     columns={"Strategy_Score": "Engine_Raw_Value"}
# )
print(f"raw_scores_df:\n{raw_scores_df.head()}\n")
print(f"engine_audit:\n{engine_audit.head()}\n")
print(f"cache_slice:\n{cache_slice}\n")
print(f"cache_slice.colums:\n{cache_slice.colums}\n")

In [ ]:
list(engine_out.perf_metrics.keys())

In [ ]:
# engine_out.perf_metrics["holding_p_gain"]
_x = engine_out.perf_metrics.get("holding_p_gain")
print(_x)

In [ ]:
# 1. Setup Parameters
decision_dt = pd.Timestamp("2025-12-10")
holding_pd = 5
lookback = 21
lookback_list = [21, 63, 189]
ticker = "NVDA"  # Let's test with the ticker you verified

# A. Run via current UI logic (Manual)
# (Imagine you selected 'Sharpe (ATRP)' in the dropdown)
ui_result = new_master_engine.run(
    EngineInput(
        mode="Rank",
        start_date=decision_dt,
        lookback_period=lookback,
        holding_period=5,
        metric="Sharpe (ATRP)",
        benchmark_ticker="SPY",
    )
)
ui_scores = ui_result.results_df["Strategy Value"]

# B. Run via new Headless Scorer
alpha_matrix = new_master_engine.compute_alpha_matrix(decision_dt, lookback)
headless_scores = alpha_matrix["Sharpe (ATRP)"].reindex(ui_scores.index)

# VERIFICATION
pd.testing.assert_series_equal(ui_scores, headless_scores, check_names=False)
print("✅ Step 1 Success: Headless Scorer matches UI logic perfectly.")

In [ ]:
# --- TEST 2: NORMALIZATION INTEGRITY ---
raw_matrix = new_master_engine.compute_alpha_matrix(decision_dt, lookback)
norm_matrix = new_master_engine.normalize_alpha_matrix(raw_matrix)

# Verification A: Mean should be near 0
mean_check = norm_matrix.mean().abs().max()
# Verification B: Std should be near 1
std_check = norm_matrix.std().max()

print(f"Max Mean Offset: {mean_check:.6f} (Target: < 0.01)")
print(f"Max Std: {std_check:.6f} (Target: ~ 1.0)")

if mean_check < 0.01 and 0.8 < std_check < 1.2:
    print("✅ Step 2 Success: The Alpha Matrix is now 'Agent-Ready'.")
else:
    print("❌ Step 2 Failed: Normalization drift detected.")

In [ ]:
# --- TEST 2.1: REGIME AWARENESS VERIFICATION ---
# decision_dt = pd.Timestamp("2025-12-10")
context = new_master_engine.compute_context_vector(decision_dt)

print("--- Agent Macro Context ---")
print(context)

# Check against your Plotly Image (Dec 10 values):
# Trend (Green line): Should be ~10% (0.10)
# Trend Vel (Orange line): Should be slightly below 0.0
# VIX Ratio: Chart says 0.86
# VIX Z (Purple line): Should be around -1.0

print(f"\nVerification Check:")
print(f"VIX Ratio in Context: {context['Context_Vix_Ratio'] + 1:.2f} (Target: 0.86)")

In [ ]:
# # --- TEST 3 (REVISITED): VERBOSE DISCOVERY VERIFICATION ---
# # decision_dt = pd.Timestamp("2025-12-10")
# # lookback = 21

# # 1. Setup 'One-Hot' Action for Sharpe (ATRP)
# registry_keys = list(METRIC_REGISTRY.keys())
# action_weights = np.zeros(len(registry_keys))
# action_weights[registry_keys.index("Sharpe (ATRP)")] = 1.0

# # 2. Run Discovery
# discovery = new_master_engine.run_discovery_action(
#     decision_dt, lookback, holding_period=5, weights=action_weights
# )

# # 3. Get UI Result for verification
# ui_result = new_master_engine.run(
#     EngineInput(
#         mode="Ranking",
#         start_date=decision_dt,
#         lookback_period=lookback,
#         holding_period=5,
#         metric="Sharpe (ATRP)",
#         benchmark_ticker="SPY",
#     )
# )

# print(f"=== DISCOVERY ENGINE VERIFICATION (Date: {decision_dt.date()}) ===")
# print(f"Target Strategy Weights: {discovery.action_weights}")
# print(f"Selected Tickers: {discovery.selected_tickers}")
# print("-" * 50)
# print(f"Internal Discovery Score (Top 1): {discovery.metric_values.iloc[0]:.4f}")
# print(
#     f"Original UI Strategy Value (Top 1): {ui_result.results_df['Strategy Value'].iloc[0]:.4f}"
# )
# print("-" * 50)
# print(f"VERITABLE REWARD (Sharpe): {discovery.veritable_reward:.4f}")
# print(f"UI REWARD (Sharpe):        {ui_result.perf_metrics['holding_p_sharpe']:.4f}")

# if discovery.selected_tickers == ui_result.tickers:
#     print("\n✅ SELECTION MATCH: The agent and UI chose the same tickers.")

In [ ]:
# --- STEP 4.1: THE VERITABLE STANDARD PROOF ---

# # 1. Setup Parameters
# decision_dt = pd.Timestamp("2025-12-10")
# holding_pd = 5
# lookback = 21
# ticker = "SHV"  # Let's test with the ticker you verified

# 2. METHOD A: The Verified UI Engine (Compounded Daily Returns)
# This is the code you already verified with Excel
ui_result = new_master_engine.run(
    EngineInput(
        mode="Manual List",
        start_date=decision_dt,
        lookback_period=lookback,
        holding_period=holding_pd,
        metric="Price Gain",
        manual_tickers=[ticker],
        benchmark_ticker="SPY",
    )
)
buy_date = ui_result.buy_date
end_date = ui_result.holding_end_date
ui_gain = ui_result.perf_metrics["holding_p_gain"]

# 3. METHOD B: The New Vectorized Shifter (Price Ratio)
# We calculate the log-gain directly from the price matrix
# log(P_end / P_start)
price_start = new_master_engine.df_close.loc[buy_date, ticker]
price_end = new_master_engine.df_close.loc[end_date, ticker]
vectorized_gain = np.log(price_end / price_start)

# 4. FINAL COMPARISON
print(f"=== VERITABLE STANDARD PROOF ({ticker}) ===")
print(f"Buy Date: {buy_date.date()} | End Date: {end_date.date()}")
print(f"Price at Buy: {price_start:.4f}")
print(f"Price at End: {price_end:.4f}")
print("-" * 40)
print(f"Method A (UI Engine Gain):   {ui_gain:.8f}")
print(f"Method B (Shifted Matrix):   {vectorized_gain:.8f}")
print("-" * 40)

diff = abs(ui_gain - vectorized_gain)
if diff < 1e-10:
    print("✅ VERIFICATION SUCCESS: Both methods are mathematically identical.")
    print("The Vectorized 'Time Machine' is now safe to use for RL Training.")
else:
    print(f"❌ VERIFICATION FAILED: Difference of {diff:.10f} detected.")

#

In [ ]:
# --- STEP 4.2: VECTORIZED DISCOVERY VERIFICATION ---

# A. Initialize the Time Machine (Do this once)
new_master_engine.precompute_reward_matrix(holding_period=5)

# decision_dt = pd.Timestamp("2025-12-10")
# lookback = 21

# B. Setup Action (Sharpe ATRP)
registry_keys = list(METRIC_REGISTRY.keys())
weights = np.zeros(len(registry_keys))
weights[registry_keys.index("Sharpe (ATRP)")] = 1.0

# C. Run Discovery (Using the NEW Vectorized functions)
# CHANGE: Only one variable on the left side
discovery = new_master_engine.run_discovery_action(
    decision_dt, lookback, holding_pd, weights=weights
)

# Access the matrix via the object property
raw_matrix = discovery.raw_alpha_matrix

print(f"SHV Raw Sharpe: {raw_matrix.loc['SHV', 'Sharpe (ATRP)']:.4f}")


# D. Get UI Result for the "Gold Standard"
ui_result = new_master_engine.run(
    EngineInput(
        mode="Ranking",
        start_date=decision_dt,
        lookback_period=lookback,
        holding_period=5,
        metric="Sharpe (ATRP)",
        benchmark_ticker="SPY",
    )
)

print(f"=== VECTORIZED ENGINE VERIFICATION (Date: {decision_dt.date()}) ===")
print(
    f"Selected Tickers: {discovery.selected_tickers[:3]}... (Match UI: {discovery.selected_tickers == ui_result.tickers})"
)
print("-" * 60)

# This confirms the 0.8381 'Signal' is preserved
print(f"SIGNAL CHECK (Lookback):")
print(f"  Discovery Score (Top 1):    {discovery.metric_values.iloc[0]:.4f}")
print(
    f"  Original UI Strategy Val:   {ui_result.results_df['Strategy Value'].iloc[0]:.4f}"
)

print("-" * 60)

# This confirms the 'Reward' matches the price action
# Note: UI Reward is Sharpe (28.29), Discovery Reward is now Total Return for RL efficiency
ui_return = ui_result.perf_metrics["holding_p_gain"]

print(f"REWARD CHECK (Holding Period Return):")
print(f"  Vectorized Reward:          {discovery.veritable_reward:.8f}")
print(f"  UI Verified Gain:           {ui_return:.8f}")

if abs(discovery.veritable_reward - ui_return) < 1e-7:
    print("\n✅ VERITABLE MATCH: The Time Machine matches the UI reality.")
    print("The Engine is now ready for High-Frequency Training.")

In [ ]:
# --- ENSEMBLE VERIFICATION TEST ---
# decision_dt = pd.Timestamp("2025-12-10")
# lookback_list = [21, 63, 189]

# 1. Generate the Ensemble
ensemble_vision = new_master_engine.compute_alpha_ensemble(decision_dt, lookback_list)

print(f"=== ENSEMBLE VISION SUMMARY (Date: {decision_dt.date()}) ===")
print(f"Total Features per Ticker: {ensemble_vision.shape[1]}")
print(f"Resolutions: {lookback_list}")
print("-" * 50)

# 2. Verify Component Integrity
# Let's check NVDA's Sharpe (ATRP) across the resolutions
nvda_stats = ensemble_vision.loc["NVDA"]

# Grab the 21d result to check against our 0.8410 benchmark
# Note: This will be the Z-scored/Clipped value, not raw 0.8410
val_21d = nvda_stats.get("21d_Sharpe (ATRP)")
val_189d = nvda_stats.get("189d_Sharpe (ATRP)")

print(f"NVDA 21d Sharpe (Z-Score):  {val_21d:.4f}")
print(f"NVDA 189d Sharpe (Z-Score): {val_189d:.4f}")

# 3. Check for Data Integrity
# Ensure we don't have all-NaN columns or duplicate data
if ensemble_vision.isna().sum().sum() == 0:
    print("\n✅ DATA INTEGRITY: Ensemble is fully populated (No NaNs).")

if val_21d != val_189d:
    print("✅ TEMPORAL DIFFERENTIATION: 21d and 189d signals are distinct.")
else:
    print("❌ ERROR: Temporal signals are identical (Slicing error?)")

# Display a sample of the 'Alpha Tensor' for one ticker
print("\nSample Feature Vector (NVDA):")
print(nvda_stats.head(10))

In [ ]:
# Check if the DYNAMIC metrics differ across resolutions
print(f"NVDA 21d Price Gain (Z):  {ensemble_vision.loc['NVDA', '21d_Price Gain']:.4f}")
print(f"NVDA 189d Price Gain (Z): {ensemble_vision.loc['NVDA', '189d_Price Gain']:.4f}")

if (
    ensemble_vision.loc["NVDA", "21d_Price Gain"]
    != ensemble_vision.loc["NVDA", "189d_Price Gain"]
):
    print(
        "\n✅ SLICING VERIFIED: Dynamic window metrics are correctly calculating different time horizons."
    )

In [ ]:
raw_matrix.loc["NVDA"]

In [ ]:
one_year_ago = decision_dt - pd.DateOffset(years=1)
one_year_ago

In [ ]:
# 1. Initialize Cache for a specific window
cache = AlphaCache(new_master_engine, lookbacks=lookback_list)

# We start in 2024 so the Agent has a year of "warm up" before 2025
one_year_ago = decision_dt - pd.DateOffset(years=1)
cache.build(start_date=one_year_ago)

# 2. Initialize the Optimized Environment
env = DiscoveryEnv(engine=new_master_engine, cache=cache, holding_period=holding_pd)

# 3. Benchmark Step Speed
import time

obs = env.reset(start_date=decision_dt)
action_size = (3 * 11) + 2
random_action = np.random.uniform(-1, 1, size=action_size)

print("\n⏱️ Starting High-Frequency Benchmark...")
start_time = time.time()
n_steps = 100

for i in range(n_steps):
    obs, reward, done, info = env.step(random_action)
    if done:
        env.reset()

end_time = time.time()
total_time = end_time - start_time

avg_step = total_time / n_steps

print(f"✅ Finished {n_steps} steps in {total_time:.4f} seconds.")
print(f"🚀 Average Step Time: {avg_step:.6f} seconds.")
print(
    f"📈 Projected Time for 1,000,000 steps: { (avg_step * 1_000_000) / 3600:.2f} hours (Previously: 9,700 hours)"
)

In [ ]:
from pathlib import Path

# 1. Define your pathing structure
# This ensures it works whether you are on a Local PC or Google Colab
cache_dir = Path("cache")
cache_file = cache_dir / "AlphaCache_2025_V1.pkl"

# 2. Safety First: Create the directory if it doesn't exist
# parents=True means it will create the whole path (e.g., data/discovery/cache)
# exist_ok=True means it won't crash if the folder is already there
cache_dir.mkdir(parents=True, exist_ok=True)

# 3. Save the 'Feature Cube'
# We use pickle because it preserves the Pandas MultiIndex and dtypes perfectly
try:
    cache.feature_cube.to_pickle(cache_file)
    print(f"✅ Persistence Successful!")
    print(f"💾 Path: {cache_file.absolute()}")
    print(f"📊 Cube Size: {cache.feature_cube.memory_usage().sum() / 1e6:.2f} MB")
except Exception as e:
    print(f"❌ Persistence Failed: {e}")

# 4. Quick Verify (The Trust Check)
# Ensure we can read it back immediately
import pandas as pd

test_load = pd.read_pickle(cache_file)
assert test_load.shape == cache.feature_cube.shape
print("🧪 Load-Back Test: PASSED. The data is safe.")

In [ ]:
# Diagnostic: List all 33 feature names
print("Current AlphaCache Feature Names:")
for i, col in enumerate(cache.feature_cube.columns):
    print(f"{i:02d}: {col}")

In [ ]:
# # This logic doesn't care if there's a space or an underscore!
# target_search = ["21d", "Sharpe", "ATRP"]
# matching_cols = [
#     c for c in cache.feature_cube.columns if all(word in c for word in target_search)
# ]

In [ ]:
# Re-initialize the env with the new class definition
env = DiscoveryEnv(new_master_engine, cache, holding_period=5)

# --- ONE-HOT PARITY TEST ---
# Search for '21d' and 'Sharpe' and '(ATRP)'
target_search = ["21d", "Sharpe", "(ATRP)"]
matching_cols = [
    c for c in cache.feature_cube.columns if all(word in c for word in target_search)
]

if not matching_cols:
    raise ValueError(
        f"❌ Search failed. Cols are: {cache.feature_cube.columns.tolist()[:5]}..."
    )

target_metric = matching_cols[0]
metric_idx = cache.feature_cube.columns.get_loc(target_metric)

# Build the One-Hot Action
action_size = cache.feature_cube.shape[1] + 2
one_hot_action = np.zeros(action_size)
one_hot_action[metric_idx] = 1.0  # Turn on ONLY this metric
one_hot_action[-2] = -1.0  # Offset = 0
one_hot_action[-1] = 1.0  # Width = 10 (Max)

# Execute Step
test_date = pd.Timestamp("2025-01-02")
env.reset(start_date=test_date)
_, _, _, info = env.step(one_hot_action)
env_tickers = info["tickers"]

# Run AlphaEngine Comparison
# Note: we pass the actual column name to see if the registry recognizes it
old_input = EngineInput(
    mode="Cascade",
    start_date=test_date,
    lookback_period=21,
    holding_period=5,
    metric="Sharpe (ATRP)",  # Standardize based on your column names
    rank_start=1,
    rank_end=10,
    benchmark_ticker="SPY",
)
old_output = new_master_engine.run(old_input)
engine_tickers = old_output.tickers

print(f"\n--- PARITY TEST: {target_metric} ---")
print(f"Environment: {env_tickers}")
print(f"AlphaEngine: {engine_tickers}")

if set(env_tickers) == set(engine_tickers):
    print("✅ PARITY ACHIEVED!")
else:
    print(
        "❌ Content mismatch. Check if AlphaEngine 'metric' name matches the feature."
    )

In [ ]:
import os
import pandas as pd
from core.environment import DiscoveryEnv

# 1. Define the Path
CACHE_PATH = os.path.join("cache", "AlphaCache_2025_V1.pkl")

# 2. Safety Check & Load
if not os.path.exists(CACHE_PATH):
    raise FileNotFoundError(f"❌ Could not find the Intelligence Cube at {CACHE_PATH}")

print(f"📂 Loading AlphaCache from {CACHE_PATH}...")
df_cube = pd.read_pickle(CACHE_PATH)

# 3. Re-inject into the Cache Object
# We pass the same lookbacks used during the 1.5-hour 'bake'
cache = AlphaCache(new_master_engine, lookbacks=[21, 63, 189])
cache.feature_cube = df_cube

print(f"✅ Intelligence Cube Restored: {cache.feature_cube.shape[0]} snapshots.")

# 4. Re-initialize the Discovery Environment
# This is our 'HFT-Speed' Training Gym
env = DiscoveryEnv(engine=new_master_engine, cache=cache, holding_period=5)

# 5. Heartbeat Verification
test_date = pd.Timestamp("2025-01-02")
obs = env.reset(start_date=test_date)
print(f"💓 System Heartbeat: Success. Current Date: {obs['date'].date()}")

In [ ]:
# --- STEP 5: OPTIMIZED DISCOVERY WALK ---
from core.environment import DiscoveryEnv

# 1. Initialize Gym using the CACHE (The high-speed version)
# Note: We don't pass 'lookbacks' here anymore because they are inside the cache!
env = DiscoveryEnv(
    engine=new_master_engine,
    cache=cache,
    holding_period=5,  # <--- This is the key change
)

# 2. Define the 'Discovery Loop'
obs = env.reset(start_date=pd.Timestamp("2025-01-02"))
done = False
total_steps = 0

print(f"🚀 Starting High-Speed Discovery Walk from {obs['date'].date()}...")

# Action Size: (Lookbacks * 11 Metrics) + 2 Rank Params
# Our cache has 33 features (3 lookbacks * 11 metrics)
action_size = cache.feature_cube.shape[1] + 2

while not done:
    # Agent outputting random weights
    random_action = np.random.uniform(-1, 1, size=action_size)

    # This call is now 10,000x faster!
    obs, reward, done, info = env.step(random_action)

    if total_steps % 10 == 0:
        print(
            f"Step {total_steps:03d} | Date: {info['date'].date()} | Reward: {reward:+.4f} | Tickers: {len(info.get('tickers', []))}"
        )

    total_steps += 1

print("-" * 50)
final_return = (env.equity_curve[-1] - 1) * 100
print(f"✅ Discovery Walk Complete. Total Steps: {total_steps}")
print(f"Final Agent Equity: {env.equity_curve[-1]:.4f} ({final_return:+.2f}%)")

In [ ]:
print(f"action_size:\n{action_size}\n")
print(f"random_action :\n{random_action }\n")
print(f"obs:\n{obs}\n")
print(f"reward:\n{reward}\n")
print(f"done:\n{done}\n")
print(f"info:\n{info}\n")
print(f"env.equity_curve:\n{env.equity_curve}\n")

In [ ]:
# 1. Pick a random sample from the cache
sample_idx = 100
sample_date = cache.feature_cube.index.get_level_values("Date")[sample_idx]
sample_ticker = cache.feature_cube.index.get_level_values("Ticker")[sample_idx]

# 2. Get the value from the Cache
cached_val = cache.feature_cube.loc[(sample_date, sample_ticker)].iloc[0]

# 3. Get the value from the Engine (The slow way)
# We calculate just for this one date
engine_ensemble = new_master_engine.compute_alpha_ensemble(sample_date, [21, 63, 252])
engine_val = engine_ensemble.loc[sample_ticker].iloc[0]

print(f"Verification for {sample_ticker} on {sample_date.date()}:")
print(f"  Cache Value:  {cached_val:.6f}")
print(f"  Engine Value: {engine_val:.6f}")
print(f"  Difference:   {abs(cached_val - engine_val):.12f}")

assert np.isclose(cached_val, engine_val, atol=1e-8), "❌ Cache Integrity Failed!"
print("✅ Cache Integrity Verified. The fast data is identical to the trusted data.")

In [ ]:
import time

# 1. Setup
lookbacks = [21, 63, 252]
cache = AlphaCache(new_master_engine, lookbacks)
cache.build()  # This takes a minute, but only runs ONCE

env = DiscoveryEnv(new_master_engine, cache)

# 2. Benchmark the Step
obs = env.reset()
action = np.random.uniform(-1, 1, size=(len(lookbacks) * 11 + 2))

start_time = time.time()
for _ in range(100):
    _, _, done, _ = env.step(action)
    if done:
        env.reset()
end_time = time.time()

avg_step_time = (end_time - start_time) / 100
print(f"⏱️ New Average Step Time: {avg_step_time:.6f} seconds")
print(f"🚀 Speedup Factor: {35 / avg_step_time:.0f}x")

In [ ]:
obs

In [ ]:
discovery

In [ ]:
# ui_result.perf_metrics
# registry_keys
# sharpe_atrp_idx
action_weights

In [ ]:
macro_df.loc[decision_dt]

In [ ]:
# universe_subset=None means "Scan the whole market"
analyzer1, stage1_pack = create_walk_forward_analyzer(
    new_master_engine, universe_subset=None
)

print("🚀 Ready for Stage 1: Run Simulation for first filter.")
analyzer1.show()

In [ ]:
# tick_price_252 = analyzer1.last_run.tickers
# tick_price_189 = analyzer1.last_run.tickers
# tick_price_126 = analyzer1.last_run.tickers
# tick_price_63 = analyzer1.last_run.tickers
# tick_price_42 = analyzer1.last_run.tickers
# tick_price_21 = analyzer1.last_run.tickers
# tick_price_10 = analyzer1.last_run.tickers

In [ ]:
# tick_sharpe_atrp_252 = analyzer1.last_run.tickers
# tick_sharpe_atrp_189 = analyzer1.last_run.tickers
# tick_sharpe_atrp_126 = analyzer1.last_run.tickers
# tick_sharpe_atrp_63 = analyzer1.last_run.tickers
# tick_sharpe_atrp_42 = analyzer1.last_run.tickers
# tick_sharpe_atrp_21 = analyzer1.last_run.tickers
# tick_sharpe_atrp_10 = analyzer1.last_run.tickers

In [ ]:
union, intersection = set_operations(
    tick_sharpe_atrp_252,
    tick_sharpe_atrp_189,
    tick_sharpe_atrp_126,
    tick_sharpe_atrp_63,
    tick_sharpe_atrp_42,
    tick_sharpe_atrp_21,
    tick_sharpe_atrp_10,
)

print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_sharpe_atrp = intersection
# int_252_189_126 = intersection

In [ ]:
print(list_to_string(int_sharpe_atrp))

In [ ]:
def list_to_string(items, separator=", ", last_separator=None):
    """
    Convert list to string with customizable separators

    Parameters:
    -----------
    items : list of strings
    separator : str, default ', '
        Separator between items
    last_separator : str, optional
        Special separator for last item (e.g., ' and ', ' or ')

    Returns:
    --------
    str : Formatted string

    Examples:
    ---------
    list_to_string(['a', 'b', 'c'])              # "a, b, c"
    list_to_string(['a', 'b', 'c'], ' | ')        # "a | b | c"
    list_to_string(['a', 'b', 'c'], ', ', ' and ')  # "a, b and c"
    """
    if not items:
        return ""

    if len(items) == 1:
        return str(items[0])

    if last_separator and len(items) > 1:
        return separator.join(items[:-1]) + last_separator + items[-1]

    return separator.join(str(item) for item in items)


# Usage examples
print(list_to_string(["a", "b", "c"]))  # a, b, c
print(list_to_string(["a", "b", "c"], " | "))  # a | b | c
print(list_to_string(["a", "b", "c"], ", ", " and "))  # a, b and c
print(list_to_string(["a", "b", "c"], ", ", ", and "))  # a, b, and c
print(list_to_string(["apple", "banana"], ", ", " and "))  # apple and banana

In [ ]:
union, intersection = set_operations(
    tick_price_252,
    tick_price_189,
    tick_price_126,
    tick_price_63,
    tick_price_42,
    tick_price_21,
    tick_price_10,
)

print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_price = intersection
# int_252_189_126 = intersection

In [ ]:
union, intersection = set_operations(
    int_sharpe_atrp,
    int_price,
)
print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_shp_atrp_price = intersection

In [ ]:
def set_operations(*lists):
    """
    Find sorted union, intersection, and elements unique to first list

    Parameters:
    -----------
    *lists : variable number of lists of strings

    Returns:
    --------
    tuple : (sorted_union, sorted_intersection, unique_to_first_list)
        - union: all unique elements from all lists
        - intersection: common elements in ALL lists
        - unique_to_first: elements only in first list, not in any other list

    Examples:
    ---------
    union, intersection, unique_first = set_operations(list1, list2)
    union, intersection, unique_first = set_operations(list1, list2, list3, list4)
    """

    if not lists:
        return [], [], []

    # Convert each list to a set
    sets = [set(lst) for lst in lists]
    first_set = sets[0]
    other_sets = sets[1:] if len(sets) > 1 else []

    # Union: all unique elements from all lists
    union_set = set().union(*sets)
    union = sorted(union_set)

    # Intersection: common elements in ALL lists
    intersection_set = first_set.intersection(*other_sets) if other_sets else first_set
    intersection = sorted(intersection_set)

    # Unique to first list: in first_set but NOT in any other set
    # Using difference: first_set - (union of all other sets)
    if other_sets:
        all_others = set().union(*other_sets)
        unique_to_first_set = (
            first_set - all_others
        )  # or first_set.difference(all_others)
    else:
        unique_to_first_set = (
            first_set  # If only one list, all elements are "unique" to it
        )

    unique_to_first = sorted(unique_to_first_set)

    return union, intersection, unique_to_first


#

In [ ]:
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# 1. LAUNCH STAGE 2 (Cascade)
# universe_subset=analyzer1.last_run.tickers means "Scan the whole market"
analyzer2, stage1_pack = create_walk_forward_analyzer(
    new_master_engine,
    universe_subset=analyzer1.last_run.tickers,
    # universe_subset=None,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

In [ ]:
###############################
###############################

In [ ]:
my_analyzer = analyzer1

my_res = SU.visualize_analyzer_structure(my_analyzer)

In [ ]:
def set_operations(list1, list2):
    """
    Find sorted union and intersection of two lists of strings

    Parameters:
    -----------
    list1, list2 : list of strings

    Returns:
    --------
    tuple : (sorted_union, sorted_intersection)
    """

    # Convert to sets for operations
    set1 = set(list1)
    set2 = set(list2)

    # Union: all unique elements from both lists
    union = sorted(set1 | set2)  # or set1.union(set2)

    # Intersection: common elements in both lists
    intersection = sorted(set1 & set2)  # or set1.intersection(set2)

    return union, intersection


# Example usage
list_a = ["apple", "banana", "cherry", "date", "elderberry"]
list_b = ["banana", "date", "fig", "grape", "apple"]

union, intersection = set_operations(list_a, list_b)

print(f"List 1: {list_a}")
print(f"List 2: {list_b}")
print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

In [ ]:
union, intersection = set_operations(list_a, list_b)

In [ ]:
tick_sharpe_trp_252 = SU.peek(4, my_res)

In [ ]:
analyzer1.last_run.tickers
analyzer1.last_run.start_date
analyzer1.last_run.holding_end_date

In [ ]:
# 3. Post-flight Reconciliation
audit = SA.verify_analyzer_short(my_analyzer)
if not audit.ok:
    print(f"🚨 ALERT: {audit.msg}")
    # You could trigger a notification or log the failure here

In [ ]:
audit = SA.verify_analyzer_long(my_analyzer, n_tickers=5)

In [ ]:
# Takes 4 seconds to run, checks selected tickers from analyzer1
SA.audit_feature_engineering_integrity(my_analyzer, mode="last_run")

### Audit Every Tickers in features_df, Takes 3 minutes 

In [ ]:
# # Takes 3 minutes to run, checks every tickers ~1580 tickers
# SA.audit_feature_engineering_integrity(my_analyzer, mode="system")

### Export Ticker's OHLCV and Features

In [ ]:
f_name_excel = OUTPUT_DIR / "Audit_Verification_Report.xlsx"

SU.export_audit_to_excel(audit_pack=my_analyzer.last_run, filename=f_name_excel)

In [ ]:
f_name_csv = OUTPUT_DIR / "all_tickers_data_stacked.csv"

# Single call replaces your 3 cells
file_path = SU.export_last_run_tickers_data_to_csv(
    analyzer=my_analyzer,
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    filename=f_name_csv,
)

In [ ]:
SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=my_analyzer.last_run.tickers,
    date_start=my_analyzer.last_run.start_date,
    date_end=analyzer1.last_run.holding_end_date,
    verbose=False,
)

In [ ]:
_tic = "NVDA"
_start_date = "2025-03-12"
_end_date = "2026-03-12"
print(features_df.loc[_tic][_start_date:_end_date])
# features_df.loc[_tic][_start_date:_end_date][["Ret_1d", "Consistency"]]

In [ ]:
result = SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=[_tic],
    date_start=_start_date,
    date_end=_end_date,
    verbose=False,
)
print(result[_tic])